In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz

# timezone UTC+8
tz = pytz.timezone('Asia/Shanghai')

# Load CSVs into memory (store full tables)
df_metric = pd.read_csv('metric.csv')
df_trace = pd.read_csv('trace.csv')
df_log = pd.read_csv('log.csv')
df_error = pd.read_csv('error_logs.csv')

# Ensure timestamp is integer and add timezone-aware datetime columns (Asia/Shanghai)
for df in (df_metric, df_trace, df_log, df_error):
    # Some CSVs might have timestamps as floats; coerce to int
    df['timestamp'] = df['timestamp'].astype(int)
    df['dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) metric.csv aggregation
g_metric = df_metric.groupby(['cmdb_id', 'kpi_name'])
metric_count_ts = g_metric['timestamp'].agg(count='count', min_timestamp='min', max_timestamp='max')
metric_p95 = g_metric['value'].quantile(0.95).rename('p95_value')
metric_agg = metric_count_ts.join(metric_p95).reset_index()

# convert min/max timestamps to timezone-aware datetimes for display
metric_agg['min_timestamp'] = pd.to_datetime(metric_agg['min_timestamp'], unit='s', utc=True).dt.tz_convert(tz)
metric_agg['max_timestamp'] = pd.to_datetime(metric_agg['max_timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# sort and limit to top 20 by count
metric_agg_display = metric_agg.sort_values('count', ascending=False)[[
    'cmdb_id', 'kpi_name', 'count', 'min_timestamp', 'max_timestamp', 'p95_value'
]].head(20)

# 2) trace.csv aggregation
g_trace = df_trace.groupby(['cmdb_id', 'trace_name'])
trace_count_ts = g_trace['timestamp'].agg(count='count', min_timestamp='min', max_timestamp='max')
trace_p95 = g_trace['value'].quantile(0.95).rename('p95_value')
trace_agg = trace_count_ts.join(trace_p95).reset_index()
trace_agg['min_timestamp'] = pd.to_datetime(trace_agg['min_timestamp'], unit='s', utc=True).dt.tz_convert(tz)
trace_agg['max_timestamp'] = pd.to_datetime(trace_agg['max_timestamp'], unit='s', utc=True).dt.tz_convert(tz)
trace_agg_display = trace_agg.sort_values('count', ascending=False)[[
    'cmdb_id', 'trace_name', 'count', 'min_timestamp', 'max_timestamp', 'p95_value'
]].head(20)

# 3) log.csv aggregation
g_log = df_log.groupby(['cmdb_id', 'log_name'])
log_count_ts = g_log['timestamp'].agg(count='count', min_timestamp='min', max_timestamp='max')
log_p95 = g_log['value'].quantile(0.95).rename('p95_value')
log_agg = log_count_ts.join(log_p95).reset_index()
log_agg['min_timestamp'] = pd.to_datetime(log_agg['min_timestamp'], unit='s', utc=True).dt.tz_convert(tz)
log_agg['max_timestamp'] = pd.to_datetime(log_agg['max_timestamp'], unit='s', utc=True).dt.tz_convert(tz)
log_agg_display = log_agg.sort_values('count', ascending=False)[[
    'cmdb_id', 'log_name', 'count', 'min_timestamp', 'max_timestamp', 'p95_value'
]].head(20)

# 4) error_logs.csv: preview up to 20 rows and total count
# Show timestamp as timezone-aware dt
error_preview = df_error[['dt', 'cmdb_id', 'message']].rename(columns={'dt': 'timestamp'}).head(20)
error_total_rows = len(df_error)

# Display compact outputs (variables)
metric_agg_display, trace_agg_display, log_agg_display, error_preview, error_total_rows

```
Out[1]:
```
Summary of the aggregated results (compact):

- All files loaded successfully and percentiles were computed globally (over each full KPI/series).

Metric aggregates (top groups by point count):
- Most top groups have 31 points spanning 2021-03-09 16:30:00+08 → 2021-03-09 17:00:00+08.
- Notable examples:
  - IG01 / JVM-Memory_..._HeapMemoryMax: count 31, p95 ≈ 2.04e9
  - IG01 / JVM-Memory_..._HeapMemoryUsed: count 31, p95 ≈ 8.75e8
  - IG01 / JVM-Threads..._ThreadCount: count 31, p95 ≈ 73
  - Tomcat04 request/session KPIs also appear in top groups (count 31) with large p95s for some request/time metrics.
  - Mysql01 and apache01 process/OS metrics appear among the top groups (count 31).

Trace aggregates (top groups by point count):
- Top trace groups mostly have 30–31 points (min timestamp 2021-03-09 16:30:00+08; max either 16:59 or 17:00+08).
- Notable examples:
  - Tomcat03 / trace.self.row_count: count 31, p95 ≈ 3,318
  - Tomcat03 / trace.self.duration_mean: count 31, p95 ≈ 0.10362 s
  - Tomcat03 / trace.self.duration_p95: count 31, p95 ≈ 1.0233 s
  - dockerA2/dockerB2/dockerA1 / various trace.to/from_MG01 and row_count entries: counts 30, p95 durations ~1.01 s or lower.

Log aggregates (top groups by point count):
- Many top log groups have 31 points (range 16:30→17:00+08); some have 30 points (ending 16:59).
- Notable examples:
  - Tomcat01 / log.row_count: count 31, p95 ≈ 475
  - apache02 / log.row_count: count 31, p95 ≈ 950
  - apache01 / log.row_count: count 31, p95 ≈ 933.5
  - Several log.error_count groups (Tomcat01, Tomcat03, apache01/02, Mysql01) have p95 = 0 (no significant error counts at the 95th percentile).
  - Some GC pause max entries for Tomcat nodes appear (counts 22–30) with p95 around 0.09–0.11 s.

Error logs:
- error_logs.csv contains 0 rows (empty). No error-log samples to show.

If you want, I can now:
- expand any of the above top-20 tables into more rows,
- filter by a specific component/time window,
- or inspect particular KPIs (e.g., the JVM memory series) in more detail.

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                           kpi_name  count             min_timestamp             max_timestamp     p95_value
1585  apache01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies     31 2021-03-09 16:30:00+08:00 2021-03-09 17:00:00+08:00  0.000000e+00
6         IG01           JVM-Threads_7778_JVM_ThreadCount_Threads     31 2021-03-09 16:30:00+08:00 2021-03-09 17:00:00+08:00  7.300000e+01
3         IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed     31 2021-03-09 16:30:00+08:00 2021-03-09 17:00:00+08:00  5.791533e+07
2         IG01          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed     31 2021-03-09 16:30:00+08:00 2021-03-09 17:00:00+08:00  8.747427e+08
0         IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax     31 2021-03-09 16:30:00+08:00 2021-03-09 17:00:00+08:00  2.040136e+09
15        IG01                             OSLinux-CPU_CPU_CPUWio     31 2021-03-09 16:30:00+08:00 2021-03-09 17:00:00+08:00  5.384290e-01
14        IG01                        OSLinux-CPU_CPU_CPUUserTime     31 2021-03-09 16:30:00+08:00 2021-03-09 17:00:00+08:00  2.536286e+01
579    Mysql01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...     31 2021-03-09 16:30:00+08:00 2021-03-09 17:00:00+08:00  4.000000e-01
580    Mysql01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount     31 2021-03-09 16:30:00+08:00 2021-03-09 17:00:00+08:00  7.000000e+00
587    Mysql01                 OSLinux-OSLinux_ZABBIX_Host_Uptime     31 2021-03-09 16:30:00+08:00 2021-03-09 17:00:00+08:00  6.478902e+06
576    Mysql01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies     31 2021-03-09 16:30:00+08:00 2021-03-09 17:00:00+08:00  0.000000e+00
1503  Tomcat04  Tomcat-Sessions_7441--UOCP_SESSIONRejectedSess...     31 2021-03-09 16:30:00+08:00 2021-03-09 17:00:00+08:00  0.000000e+00
68        IG01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT     31 2021-03-09 16:30:00+08:00 2021-03-09 17:00:00+08:00  0.000000e+00
1497  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...     31 2021-03-09 16:30:00+08:00 2021-03-09 17:00:00+08:00  2.740000e+02
1498  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...     31 2021-03-09 16:30:00+08:00 2021-03-09 17:00:00+08:00  3.248520e+05
1499  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_Processin...     31 2021-03-09 16:30:00+08:00 2021-03-09 17:00:00+08:00  1.650920e+08
1500  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...     31 2021-03-09 16:30:00+08:00 2021-03-09 17:00:00+08:00  1.686075e+05
1501  Tomcat04    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter     31 2021-03-09 16:30:00+08:00 2021-03-09 17:00:00+08:00  0.000000e+00
1502  Tomcat04  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...     31 2021-03-09 16:30:00+08:00 2021-03-09 17:00:00+08:00  0.000000e+00
81        IG01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...     31 2021-03-09 16:30:00+08:00 2021-03-09 17:00:00+08:00  3.000000e-01,       cmdb_id                     trace_name  count             min_timestamp             max_timestamp    p95_value
167  Tomcat03           trace.self.row_count     31 2021-03-09 16:30:00+08:00 2021-03-09 17:00:00+08:00  3318.000000
165  Tomcat03       trace.self.duration_mean     31 2021-03-09 16:30:00+08:00 2021-03-09 17:00:00+08:00     0.103620
166  Tomcat03        trace.self.duration_p95     31 2021-03-09 16:30:00+08:00 2021-03-09 17:00:00+08:00     1.023300
259  dockerB2     trace.to_MG01.duration_p95     30 2021-03-09 16:30:00+08:00 2021-03-09 16:59:00+08:00     1.012955
220  dockerA2        trace.self.duration_p95     30 2021-03-09 16:30:00+08:00 2021-03-09 16:59:00+08:00     1.018000
221  dockerA2           trace.self.row_count     30 2021-03-09 16:30:00+08:00 2021-03-09 16:59:00+08:00  3081.650000
258  dockerB2    trace.to_MG01.duration_mean     30 2021-03-09 16:30:00+08:00 2021-03-09 16:59:00+08:00     0.099025
219  dockerA2       trace.self.duration_mean     30 2021-03-09 16:30:00+08:00 2021-03-09 16:59:00+08:00     0.105980
260  dockerB2        trace.to_MG01.row_count     30 2021-03-09 16:30:00+08:00 2021-03-09 16:59:00+08:00   433.450000
211  dockerA2   trace.from_MG01.duration_p95     30 2021-03-09 16:30:00+08:00 2021-03-09 16:59:00+08:00     2.881925
212  dockerA2      trace.from_MG01.row_count     30 2021-03-09 16:30:00+08:00 2021-03-09 16:59:00+08:00   365.150000
203  dockerA1           trace.self.row_count     30 2021-03-09 16:30:00+08:00 2021-03-09 16:59:00+08:00  2980.550000
204  dockerA1    trace.to_MG01.duration_mean     30 2021-03-09 16:30:00+08:00 2021-03-09 16:59:00+08:00     0.134508
205  dockerA1     trace.to_MG01.duration_p95     30 2021-03-09 16:30:00+08:00 2021-03-09 16:59:00+08:00     1.017545
206  dockerA1        trace.to_MG01.row_count     30 2021-03-09 16:30:00+08:00 2021-03-09 16:59:00+08:00   110.300000
210  dockerA2  trace.from_MG01.duration_mean     30 2021-03-09 16:30:00+08:00 2021-03-09 16:59:00+08:00     0.768197
201  dockerA1       trace.self.duration_mean     30 2021-03-09 16:30:00+08:00 2021-03-09 16:59:00+08:00     0.053018
202  dockerA1        trace.self.duration_p95     30 2021-03-09 16:30:00+08:00 2021-03-09 16:59:00+08:00     0.713825
59       MG01  trace.from_dockerB2.row_count     30 2021-03-09 16:30:00+08:00 2021-03-09 16:59:00+08:00   433.450000
63       MG01       trace.self.duration_mean     30 2021-03-09 16:30:00+08:00 2021-03-09 16:59:00+08:00     0.393161,      cmdb_id          log_name  count             min_timestamp             max_timestamp  p95_value
4   Tomcat01     log.row_count     31 2021-03-09 16:30:00+08:00 2021-03-09 17:00:00+08:00   475.0000
2   Tomcat01   log.error_count     31 2021-03-09 16:30:00+08:00 2021-03-09 17:00:00+08:00     0.0000
17  apache02     log.row_count     31 2021-03-09 16:30:00+08:00 2021-03-09 17:00:00+08:00   950.0000
15  apache01     log.row_count     31 2021-03-09 16:30:00+08:00 2021-03-09 17:00:00+08:00   933.5000
8   Tomcat03   log.error_count     31 2021-03-09 16:30:00+08:00 2021-03-09 17:00:00+08:00     0.0000
16  apache02   log.error_count     31 2021-03-09 16:30:00+08:00 2021-03-09 17:00:00+08:00     0.0000
14  apache01   log.error_count     31 2021-03-09 16:30:00+08:00 2021-03-09 17:00:00+08:00     0.0000
13  Tomcat04     log.row_count     31 2021-03-09 16:30:00+08:00 2021-03-09 17:00:00+08:00   463.5000
11  Tomcat04   log.error_count     31 2021-03-09 16:30:00+08:00 2021-03-09 17:00:00+08:00     0.0000
10  Tomcat03     log.row_count     31 2021-03-09 16:30:00+08:00 2021-03-09 17:00:00+08:00   504.5000
6   Tomcat02  log.gc_pause_max     30 2021-03-09 16:30:00+08:00 2021-03-09 16:59:00+08:00     0.0900
7   Tomcat02     log.row_count     30 2021-03-09 16:30:00+08:00 2021-03-09 16:59:00+08:00     2.0000
5   Tomcat02   log.error_count     30 2021-03-09 16:30:00+08:00 2021-03-09 16:59:00+08:00     0.0000
0    Mysql01   log.error_count     26 2021-03-09 16:30:00+08:00 2021-03-09 16:59:00+08:00     0.0000
1    Mysql01     log.row_count     26 2021-03-09 16:30:00+08:00 2021-03-09 16:59:00+08:00    13.2500
3   Tomcat01  log.gc_pause_max     22 2021-03-09 16:30:00+08:00 2021-03-09 16:58:00+08:00     0.0900
9   Tomcat03  log.gc_pause_max     22 2021-03-09 16:30:00+08:00 2021-03-09 16:59:00+08:00     0.1095
12  Tomcat04  log.gc_pause_max     22 2021-03-09 16:30:00+08:00 2021-03-09 16:59:00+08:00     0.0995, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [], 0)```
```

In [ ]:
# In[2]:

# reuse existing variables: df_metric, tz
import numpy as np
import pandas as pd

# 1) compute global P95 per (cmdb_id, kpi_name) over full series (do NOT filter first)
global_p95 = df_metric.groupby(['cmdb_id', 'kpi_name'])['value'].quantile(0.95).rename('global_p95').reset_index()

# 2) filter metric rows to the incident window in UTC+8
start = pd.Timestamp('2021-03-09 16:30:00', tz=tz)
end = pd.Timestamp('2021-03-09 17:00:00', tz=tz)
# ensure 'dt' exists in df_metric
if 'dt' not in df_metric.columns:
    df_metric['dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

df_win = df_metric[(df_metric['dt'] >= start) & (df_metric['dt'] <= end)].copy()

# 3) join global_p95 into the windowed dataframe
df_win = df_win.merge(global_p95, on=['cmdb_id', 'kpi_name'], how='left')

# 4) mark anomaly points where value >= global_p95
df_win['anomaly'] = df_win['value'] >= df_win['global_p95']

# 5) aggregate per (cmdb_id, kpi_name) in the window
agg_window = df_win.groupby(['cmdb_id', 'kpi_name']).agg(
    total_points_in_window = ('value', 'count'),
    anomaly_point_count = ('anomaly', 'sum'),
    max_value_in_window = ('value', 'max')
).reset_index()

# 6) earliest anomaly timestamp (if any)
earliest_anom = df_win[df_win['anomaly']].groupby(['cmdb_id', 'kpi_name'])['dt'].min().rename('earliest_anomaly_timestamp').reset_index()

# 7) combine with global_p95 and earliest anomaly
summary = agg_window.merge(global_p95, on=['cmdb_id','kpi_name'], how='left').merge(earliest_anom, on=['cmdb_id','kpi_name'], how='left')

# 8) compute severity_ratio = (max_value_in_window - global_p95)/global_p95
# handle global_p95 == 0 -> set to np.inf
def compute_severity(row):
    gp = row['global_p95']
    mv = row['max_value_in_window']
    if pd.isna(gp):
        return np.nan
    if gp == 0:
        return np.inf
    return (mv - gp) / gp

summary['severity_ratio'] = summary.apply(compute_severity, axis=1)

# 9) tidy columns and sort by severity_ratio desc then anomaly_point_count desc
summary_display = summary[[
    'cmdb_id', 'kpi_name', 'global_p95', 'total_points_in_window',
    'anomaly_point_count', 'earliest_anomaly_timestamp', 'max_value_in_window', 'severity_ratio'
]].sort_values(['severity_ratio','anomaly_point_count'], ascending=[False, False]).head(20)

# Keep output compact
summary_display

```
Out[2]:
```
# Summarize the previous execution results as a single plain-English string
import numpy as np

# Use the previously computed `summary_display` (top 20 rows). If not present, raise helpful error.
if 'summary_display' not in globals():
    raise RuntimeError("summary_display not found in kernel state. Re-run the previous aggregation step first.")

shown = summary_display.copy()
total_shown = len(shown)
all_anomalies = int((shown['anomaly_point_count'] == shown['total_points_in_window']).sum())
inf_severity = int((shown['severity_ratio'] == np.inf).sum())

lines = []
lines.append("Summary of metric anomaly scan (window 2021-03-09 16:30:00+08 → 2021-03-09 17:00:00+08):")
lines.append(f"- Top rows shown: {total_shown}")
lines.append(f"- Groups where every point in the window >= global P95: {all_anomalies} (of top {total_shown})")
lines.append(f"- Groups with global_p95 == 0 (severity_ratio reported as infinity): {inf_severity}")
lines.append("- Note: severity_ratio = (max_value_in_window - global_p95) / global_p95; infinite when global_p95 == 0.")
lines.append("")
lines.append("Top example groups (up to 5) from the displayed table:")
for _, r in shown.head(5).iterrows():
    lines.append(f"  - {r['cmdb_id']} | {r['kpi_name']} | global_p95={r['global_p95']} | "
                 f"points={int(r['total_points_in_window'])} | anomalies={int(r['anomaly_point_count'])} | "
                 f"earliest_anomaly={r['earliest_anomaly_timestamp']} | max={r['max_value_in_window']} | "
                 f"severity_ratio={r['severity_ratio']}")
lines.append("")
lines.append("Interpretation / quick guidance:")
lines.append("- Many top flagged metrics have global P95 == 0, causing infinite severity ratios; these are often counters or sparse indicators (e.g., TCP state counts or session counters).")
lines.append("- Several network TCP metrics (e.g., CLOSE-WAIT, FIN-WAIT) and Tomcat session-related KPIs show anomalies across all points in the window.")
lines.append("- Recommendation: treat zero-baseline KPIs carefully (use a different rule than simple ratio), and inspect the highlighted components (IG01, Mysql01/Mysql02, Tomcat01/02, Redis01) for further context (logs/traces) before acting.")

summary_text = "\n".join(lines)
summary_text

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  global_p95  total_points_in_window  anomaly_point_count earliest_anomaly_timestamp  max_value_in_window  severity_ratio
67        IG01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT         0.0                      31                   31  2021-03-09 16:30:00+08:00                  1.0             inf
68        IG01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT         0.0                      31                   31  2021-03-09 16:30:00+08:00                  0.0             inf
573    Mysql01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT         0.0                      31                   31  2021-03-09 16:30:00+08:00                  0.0             inf
574    Mysql01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT         0.0                      31                   31  2021-03-09 16:30:00+08:00                  0.0             inf
576    Mysql01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies         0.0                      31                   31  2021-03-09 16:30:00+08:00                  0.0             inf
799    Mysql02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT         0.0                      31                   31  2021-03-09 16:30:00+08:00                  0.0             inf
800    Mysql02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT         0.0                      31                   31  2021-03-09 16:30:00+08:00                  0.0             inf
802    Mysql02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies         0.0                      31                   31  2021-03-09 16:30:00+08:00                  0.0             inf
874    Redis01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT         0.0                      31                   31  2021-03-09 16:30:00+08:00                  1.0             inf
1175  Tomcat01    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter         0.0                      31                   31  2021-03-09 16:30:00+08:00                  0.0             inf
1176  Tomcat01  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...         0.0                      31                   31  2021-03-09 16:30:00+08:00                  0.0             inf
1177  Tomcat01  Tomcat-Sessions_7441--UOCP_SESSIONRejectedSess...         0.0                      31                   31  2021-03-09 16:30:00+08:00                  0.0             inf
1178  Tomcat01        Tomcat-Sessions_7441--_SESSIONActiveCounter         0.0                      31                   31  2021-03-09 16:30:00+08:00                  0.0             inf
1179  Tomcat01     Tomcat-Sessions_7441--_SESSIONKeepaliveCounter         0.0                      31                   31  2021-03-09 16:30:00+08:00                  0.0             inf
1180  Tomcat01     Tomcat-Sessions_7441--_SESSIONRejectedSessions         0.0                      31                   31  2021-03-09 16:30:00+08:00                  0.0             inf
1181  Tomcat01  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...         0.0                      31                   31  2021-03-09 16:30:00+08:00                  0.0             inf
1182  Tomcat01  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...         0.0                      31                   31  2021-03-09 16:30:00+08:00                  0.0             inf
1183  Tomcat01  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...         0.0                      31                   31  2021-03-09 16:30:00+08:00                  0.0             inf
1281  Tomcat02    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter         0.0                      31                   31  2021-03-09 16:30:00+08:00                  0.0             inf
1282  Tomcat02  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...         0.0                      31                   31  2021-03-09 16:30:00+08:00                  0.0             inf```
```

In [ ]:
# In[3]:

# Reuse existing variables: df_metric, global_p95, tz
import pandas as pd
import numpy as np

# Ensure timezone-aware datetime column exists
if 'dt' not in df_metric.columns:
    df_metric['dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Define incident window in UTC+8
start = pd.Timestamp('2021-03-09 16:30:00', tz=tz)
end = pd.Timestamp('2021-03-09 17:00:00', tz=tz)

# Windowed rows from metric (do NOT recompute global P95 here)
df_win_pos = df_metric[(df_metric['dt'] >= start) & (df_metric['dt'] <= end)].merge(
    global_p95, on=['cmdb_id', 'kpi_name'], how='left'
)

# Consider only component-KPIs whose global_p95 > 0
df_win_pos = df_win_pos[df_win_pos['global_p95'] > 0].copy()

# Mark anomaly points where value >= global_p95
df_win_pos['anomaly'] = df_win_pos['value'] >= df_win_pos['global_p95']

# Aggregate required metrics per (cmdb_id, kpi_name)
agg = df_win_pos.groupby(['cmdb_id', 'kpi_name']).agg(
    total_points_in_window = ('value', 'count'),
    anomaly_point_count = ('anomaly', 'sum'),
    max_value_in_window = ('value', 'max')
).reset_index()

# Earliest anomaly timestamp per group (if any)
earliest_anom = df_win_pos[df_win_pos['anomaly']].groupby(
    ['cmdb_id', 'kpi_name']
)['dt'].min().rename('earliest_anomaly_timestamp').reset_index()

# Combine with global_p95 and earliest anomaly
summary_positive = agg.merge(global_p95, on=['cmdb_id','kpi_name'], how='left').merge(
    earliest_anom, on=['cmdb_id','kpi_name'], how='left'
)

# Compute severity_ratio safely (global_p95 > 0 here)
summary_positive['severity_ratio'] = (summary_positive['max_value_in_window'] - summary_positive['global_p95']) / summary_positive['global_p95']

# Select and sort as requested
summary_positive_display = summary_positive[[
    'cmdb_id', 'kpi_name', 'global_p95', 'total_points_in_window',
    'anomaly_point_count', 'earliest_anomaly_timestamp', 'max_value_in_window', 'severity_ratio'
]].sort_values(['severity_ratio', 'anomaly_point_count'], ascending=[False, False]).head(20)

# Display compact result
summary_positive_display

```
Out[3]:
```
# Summarize the top metric anomalies (plain English) using the existing `summary_positive_display`
import numpy as np

if 'summary_positive_display' not in globals():
    raise RuntimeError("summary_positive_display not found. Re-run the aggregation step first.")

df = summary_positive_display.copy()
n_rows = len(df)
max_sev = df['severity_ratio'].replace([np.inf, -np.inf], np.nan).max()
top_examples = df.head(5)

lines = []
lines.append("Summary of metric anomalies (global P95 > 0) within 2021-03-09 16:30 → 17:00 +08:")
lines.append(f"- Returned groups (max 20): {n_rows}")
lines.append(f"- Highest severity_ratio observed in the returned set: {max_sev:.3f}")
lines.append("- Most flagged groups show a single anomalous point in the window with a large spike vs global P95.")
lines.append("")
lines.append("Top example groups (up to 5):")
for _, r in top_examples.iterrows():
    lines.append(
        f"  - {r['cmdb_id']} | {r['kpi_name']} | global_p95={r['global_p95']}"
        f" | points={int(r['total_points_in_window'])} | anomalies={int(r['anomaly_point_count'])}"
        f" | earliest_anomaly={r['earliest_anomaly_timestamp']} | max={r['max_value_in_window']}"
        f" | severity_ratio={r['severity_ratio']:.6g}"
    )
lines.append("")
lines.append("Quick interpretation and recommendation:")
lines.append("- The top spikes are on MG02 (TCP-FIN-WAIT), several MySQL KPIs, Tomcat network TCP states, and some disk I/O metrics.")
lines.append("- Many cases have severity_ratio ≈ 19, meaning the max value is about 20× the global P95 (a strong spike).")
lines.append("- Action: investigate the listed timestamps/components (logs/traces on those servers) for the single large spikes; check whether these are transient bursts or indicate resource saturation.")
lines.append("- Note: this table only includes metrics whose global P95 > 0, and anomaly detection used value >= global P95 (global percentile computed over the full series).")

summary_text = "\n".join(lines)
summary_text

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  global_p95  total_points_in_window  anomaly_point_count earliest_anomaly_timestamp  max_value_in_window  severity_ratio
295       MG02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT    0.500000                      20                    1  2021-03-09 16:56:00+08:00            10.000000       19.000000
311    Mysql01                        Mysql-MySQL_3306_Com Insert    0.013335                      20                    1  2021-03-09 16:30:00+08:00             0.266700       19.000000
457    Mysql02          Mysql-MySQL_3306_CurrentSQLMaxRunningTime    0.050000                      20                    1  2021-03-09 16:43:00+08:00             1.000000       19.000000
490    Mysql02              Mysql-MySQL_3306_LongestTrxActiveTime    0.050000                      20                    1  2021-03-09 16:43:00+08:00             1.000000       19.000000
1023  Tomcat03     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    0.050000                      20                    1  2021-03-09 17:00:00+08:00             1.000000       19.000000
1112  Tomcat04     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    0.050000                      20                    1  2021-03-09 16:31:00+08:00             1.000000       19.000000
215       MG01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    0.100000                      19                    1  2021-03-09 16:52:00+08:00             1.000000        9.000000
691    Redis02                             OSLinux-CPU_CPU_CPUWio    0.038430                      30                    2  2021-03-09 16:32:00+08:00             0.242600        5.312776
576    Mysql02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps    1.250000                      16                    1  2021-03-09 16:44:00+08:00             5.000000        3.000000
91        IG02                             OSLinux-CPU_CPU_CPUWio    0.737407                      29                    2  2021-03-09 16:30:00+08:00             2.735398        2.709482
47        IG01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead    1.920000                      15                    1  2021-03-09 16:50:00+08:00             6.400000        2.333333
419    Mysql01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...    0.240000                      15                    1  2021-03-09 16:43:00+08:00             0.800000        2.333333
578    Mysql02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead    0.520000                      15                    1  2021-03-09 17:00:00+08:00             1.733333        2.333333
1140  apache01                    OSLinux-CPU_CPU-1_SingleCpuidle   20.152672                      15                    1  2021-03-09 16:34:00+08:00            67.175573        2.333333
46        IG01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps    0.065000                      15                    1  2021-03-09 16:50:00+08:00             0.216667        2.333333
131       IG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps    0.005010                      15                    1  2021-03-09 16:50:00+08:00             0.016700        2.333333
132       IG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead    0.039990                      15                    1  2021-03-09 16:50:00+08:00             0.133300        2.333333
568    Mysql02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...    0.840000                      15                    1  2021-03-09 16:36:00+08:00             2.800000        2.333333
577    Mysql02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps    0.065000                      15                    1  2021-03-09 17:00:00+08:00             0.216667        2.333333
1246  apache02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...    1.200000                      15                    1  2021-03-09 16:48:00+08:00             4.000000        2.333333```
```

In [ ]:
# In[4]:

# Reuse existing variables: df_trace, tz
import pandas as pd
import numpy as np

# Ensure df_trace is loaded and has timezone-aware datetime column
if 'df_trace' not in globals():
    df_trace = pd.read_csv('trace.csv')
if 'dt' not in df_trace.columns:
    df_trace['timestamp'] = df_trace['timestamp'].astype(int)
    df_trace['dt'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Compute global P95 per (cmdb_id, trace_name) over the full series (do NOT filter first)
global_p95_trace = df_trace.groupby(['cmdb_id', 'trace_name'])['value'].quantile(0.95).rename('global_p95').reset_index()

# 2) Filter trace rows to the incident window in UTC+8
start = pd.Timestamp('2021-03-09 16:30:00', tz=tz)
end = pd.Timestamp('2021-03-09 17:00:00', tz=tz)
df_trace_win = df_trace[(df_trace['dt'] >= start) & (df_trace['dt'] <= end)].copy()

# 3) Join global P95 into the windowed dataframe
df_trace_win = df_trace_win.merge(global_p95_trace, on=['cmdb_id', 'trace_name'], how='left')

# 4) Mark anomaly points where value >= global_p95
df_trace_win['anomaly'] = df_trace_win['value'] >= df_trace_win['global_p95']

# 5) Aggregate per (cmdb_id, trace_name) in the window
agg_trace = df_trace_win.groupby(['cmdb_id', 'trace_name']).agg(
    total_points_in_window = ('value', 'count'),
    anomaly_point_count = ('anomaly', 'sum'),
    max_value_in_window = ('value', 'max')
).reset_index()

# 6) earliest anomaly timestamp (if any)
earliest_anom_trace = df_trace_win[df_trace_win['anomaly']].groupby(
    ['cmdb_id', 'trace_name']
)['dt'].min().rename('earliest_anomaly_timestamp').reset_index()

# 7) combine with global_p95 and earliest anomaly
summary_trace = agg_trace.merge(global_p95_trace, on=['cmdb_id','trace_name'], how='left').merge(
    earliest_anom_trace, on=['cmdb_id','trace_name'], how='left'
)

# 8) compute severity_ratio; handle global_p95 == 0 -> inf
summary_trace['severity_ratio'] = np.where(
    summary_trace['global_p95'] == 0,
    np.inf,
    (summary_trace['max_value_in_window'] - summary_trace['global_p95']) / summary_trace['global_p95']
)

# 9) Select requested columns and sort
summary_trace_display = summary_trace[[
    'cmdb_id', 'trace_name', 'global_p95', 'total_points_in_window',
    'anomaly_point_count', 'earliest_anomaly_timestamp', 'max_value_in_window', 'severity_ratio'
]].sort_values(['severity_ratio', 'anomaly_point_count'], ascending=[False, False]).head(20)

# Display compact result
summary_trace_display

```
Out[4]:
```
# Summarize the trace anomaly results as a plain-English string (using existing `summary_trace_display`)
import numpy as np

if 'summary_trace_display' not in globals():
    raise RuntimeError("summary_trace_display not found in kernel state. Re-run the trace aggregation step first.")

df = summary_trace_display.copy()
lines = []
lines.append("Summary of trace anomaly scan (window 2021-03-09 16:30 → 17:00 +08):")
lines.append(f"- Returned groups (top shown, max 20): {len(df)}")
lines.append("- Highest severity_ratio in the displayed set: {:.3f}".format(df['severity_ratio'].replace([np.inf, -np.inf], np.nan).max()))
lines.append("")
lines.append("Top example trace anomalies (up to 5):")
for _, r in df.head(5).iterrows():
    lines.append(
        f"  - {r['cmdb_id']} | {r['trace_name']} | global_p95={r['global_p95']}"
        f" | points={int(r['total_points_in_window'])} | anomalies={int(r['anomaly_point_count'])}"
        f" | earliest_anomaly={r['earliest_anomaly_timestamp']} | max={r['max_value_in_window']}"
        f" | severity_ratio={r['severity_ratio']:.6g}"
    )

lines.append("")
lines.append("Key takeaways:")
lines.append("- The strongest relative increase is MG01 trace.from_root.duration_mean (≈70% above its global P95).")
lines.append("- Other notable increases (~67% above P95) involve MG02/dockerA2 communication traces and several MG↔docker paths.")
lines.append("- Most flagged traces have only 1–2 anomalous points in the window (transient spikes rather than sustained degradation).")
lines.append("- Affected components include MG01, MG02, dockerA1/A2, dockerB1/B2, IG01 and Tomcat nodes—indicating localized latency spikes in inter-service calls.")
lines.append("")
lines.append("Recommended next steps:")
lines.append("- Inspect traces and related logs around the reported earliest_anomaly timestamps for MG01/MG02 and the docker nodes.")
lines.append("- Correlate with server metrics (CPU, disk I/O, network) and application logs to determine if these are transient network/queueing blips or symptoms of resource contention.")
lines.append("- If recurring, consider deeper sampling or increased tracing around the MG↔docker and IG↔Tomcat call paths.")

summary_text = "\n".join(lines)
summary_text

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                         trace_name  global_p95  total_points_in_window  anomaly_point_count earliest_anomaly_timestamp  max_value_in_window  severity_ratio
60       MG01      trace.from_root.duration_mean    1.221204                      14                    1  2021-03-09 16:53:00+08:00             2.082500        0.705285
93       MG02  trace.from_dockerA2.duration_mean    0.132121                      27                    2  2021-03-09 16:32:00+08:00             0.221250        0.674598
225  dockerA2        trace.to_MG02.duration_mean    0.132121                      27                    2  2021-03-09 16:32:00+08:00             0.221250        0.674598
50       MG01      trace.from_dockerA1.row_count  110.300000                      30                    2  2021-03-09 16:58:00+08:00           163.000000        0.477788
206  dockerA1            trace.to_MG01.row_count  110.300000                      30                    2  2021-03-09 16:58:00+08:00           163.000000        0.477788
202  dockerA1            trace.self.duration_p95    0.713825                      30                    2  2021-03-09 16:39:00+08:00             1.008000        0.412111
54       MG01  trace.from_dockerB1.duration_mean    0.096962                      30                    2  2021-03-09 16:53:00+08:00             0.134360        0.385690
240  dockerB1        trace.to_MG01.duration_mean    0.096962                      30                    2  2021-03-09 16:53:00+08:00             0.134360        0.385690
12       IG01    trace.to_Tomcat03.duration_mean    0.713833                      30                    2  2021-03-09 16:31:00+08:00             0.960089        0.344978
156  Tomcat03      trace.from_IG01.duration_mean    0.713833                      30                    2  2021-03-09 16:31:00+08:00             0.960089        0.344978
4        IG01            trace.self.duration_p95    3.112627                      30                    2  2021-03-09 16:39:00+08:00             4.124000        0.324926
7        IG01     trace.to_Tomcat01.duration_p95    3.162478                      30                    2  2021-03-09 16:39:00+08:00             4.172500        0.319377
121  Tomcat01       trace.from_IG01.duration_p95    3.162478                      30                    2  2021-03-09 16:39:00+08:00             4.172500        0.319377
48       MG01  trace.from_dockerA1.duration_mean    0.134508                      30                    2  2021-03-09 16:39:00+08:00             0.177167        0.317144
204  dockerA1        trace.to_MG01.duration_mean    0.134508                      30                    2  2021-03-09 16:39:00+08:00             0.177167        0.317144
117      MG02    trace.to_dockerB2.duration_mean    0.583558                      27                    2  2021-03-09 16:39:00+08:00             0.759433        0.301384
249  dockerB2      trace.from_MG02.duration_mean    0.583558                      27                    2  2021-03-09 16:39:00+08:00             0.759433        0.301384
31       IG02     trace.to_Tomcat03.duration_p95    3.101880                      30                    2  2021-03-09 16:48:00+08:00             4.005000        0.291152
160  Tomcat03       trace.from_IG02.duration_p95    3.101880                      30                    2  2021-03-09 16:48:00+08:00             4.005000        0.291152
16       IG01     trace.to_Tomcat04.duration_p95    3.152827                      30                    2  2021-03-09 16:39:00+08:00             4.045800        0.283229```
```

In [ ]:
# In[5]:

# Reuse existing df_trace and tz if available; otherwise load and prepare
import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

if 'df_trace' not in globals():
    df_trace = pd.read_csv('trace.csv')

# Ensure timestamp is int and dt column (timezone-aware UTC+8) exists
df_trace['timestamp'] = df_trace['timestamp'].astype(int)
df_trace['dt'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Compute global P95 per (cmdb_id, trace_name) over the full series (do NOT filter first)
global_p95_trace = df_trace.groupby(['cmdb_id', 'trace_name'], as_index=False)['value'].quantile(0.95).rename(columns={'value':'global_p95'})

# 2) Filter trace rows to the incident window in UTC+8
start = pd.Timestamp('2021-03-09 16:30:00', tz=tz)
end = pd.Timestamp('2021-03-09 17:00:00', tz=tz)
df_trace_win = df_trace[(df_trace['dt'] >= start) & (df_trace['dt'] <= end)].copy()

# 3) Restrict to the given components
components = ['MG01','MG02','Mysql01','Mysql02','Tomcat03','Tomcat04','IG01','dockerA1','dockerA2','dockerB1','dockerB2']
df_trace_win = df_trace_win[df_trace_win['cmdb_id'].isin(components)].copy()

# 4) Join global_p95 (computed globally) into the windowed dataframe
df_trace_win = df_trace_win.merge(global_p95_trace, on=['cmdb_id','trace_name'], how='left')

# 5) Identify rows where value >= global_p95 (note: global_p95 computed on full series)
# If global_p95 is NaN (no global series), treat as no anomaly (exclude)
df_trace_win['is_anomaly'] = False
mask_has_p95 = df_trace_win['global_p95'].notna()
df_trace_win.loc[mask_has_p95, 'is_anomaly'] = df_trace_win.loc[mask_has_p95, 'value'] >= df_trace_win.loc[mask_has_p95, 'global_p95']

# 6) Detail table: rows where is_anomaly == True
detail_df = df_trace_win[df_trace_win['is_anomaly']].copy()
# Prepare output columns: cmdb_id, trace_name, timestamp (dt), value
detail_df_output = detail_df[['cmdb_id', 'trace_name', 'dt', 'value']].rename(columns={'dt':'timestamp'})

# Limit to max 50 rows for display; sort for readability by cmdb_id, trace_name, timestamp
detail_df_display = detail_df_output.sort_values(['cmdb_id','trace_name','timestamp']).head(50)

# 7) Summary table: one row per listed component with total_anomaly_rows_in_window and earliest_anomaly_timestamp
summary = detail_df.groupby('cmdb_id').agg(
    total_anomaly_rows_in_window = ('is_anomaly','sum'),
    earliest_anomaly_timestamp = ('dt','min')
).reset_index()

# Ensure every listed component appears in the summary (include zeros)
summary_all = pd.DataFrame({'cmdb_id': components})
summary_all = summary_all.merge(summary, on='cmdb_id', how='left')
summary_all['total_anomaly_rows_in_window'] = summary_all['total_anomaly_rows_in_window'].fillna(0).astype(int)
# earliest_anomaly_timestamp stays NaT if none
summary_all = summary_all[['cmdb_id','total_anomaly_rows_in_window','earliest_anomaly_timestamp']]

# Display compact results (detail up to 50 rows, and one-row-per-component summary)
detail_df_display, summary_all

```
Out[5]:
```
# Summarize the trace anomaly findings in plain English as a single string
import pandas as pd
import numpy as np

# Use existing summary_all and detail_df_display from kernel state
if 'summary_all' not in globals() or 'detail_df_display' not in globals():
    raise RuntimeError("Required results not found in kernel state. Re-run the anomaly extraction step first.")

s = summary_all.copy()

# Key highlights
rows = []
rows.append("Trace anomaly summary (global P95 computed on full series; anomalies are window rows >= that P95).")
rows.append("Window: 2021-03-09 16:30:00+08 → 2021-03-09 17:00:00+08. Components restricted to the given list.")
rows.append("")

# Component-level summary (counts and earliest timestamps)
rows.append("Anomaly counts and earliest occurrence per component:")
for _, r in s.iterrows():
    ts = r['earliest_anomaly_timestamp']
    ts_str = ts.isoformat() if pd.notna(ts) else "none"
    rows.append(f"- {r['cmdb_id']}: total_anomaly_rows_in_window={int(r['total_anomaly_rows_in_window'])}, earliest_anomaly_timestamp={ts_str}")

rows.append("")
# Highlight top components
top_by_count = s.sort_values('total_anomaly_rows_in_window', ascending=False).head(5)
top_list = ", ".join([f"{row.cmdb_id}({row.total_anomaly_rows_in_window})" for _, row in top_by_count.iterrows()])
rows.append(f"Top components by anomaly row count: {top_list}")

rows.append("")
rows.append("Notable observations:")
rows.append("- MG02 and MG01 have the largest numbers of anomalous trace rows (MG02: 84, MG01: 81), with earliest anomalies at ~16:30–16:32.")
rows.append("- dockerB1/dockers/dockerA2 also show many anomaly rows (dockerB1: 44, dockerA2: 36).")
rows.append("- Tomcat03, Tomcat04, IG01, dockerA1 and dockerB2 each have ~33 anomaly rows, often starting at ~16:30–16:31.")
rows.append("- Mysql01 and Mysql02 had no trace rows flagged as >= global P95 in the window.")
rows.append("- The detailed table (sample shown separately) includes up to 50 anomaly rows across these components covering trace.duration and trace.row_count metrics; many anomalies are clustered around 16:30–16:58.")

rows.append("")
rows.append("Quick recommendation:")
rows.append("- Investigate MG01/MG02 and the docker nodes first (check inter-service latency, network, and resource metrics around the reported earliest times).")
rows.append("- Correlate these trace spikes with application logs and system metrics to determine if these are transient spikes or indicate systemic degradation.")

summary_text = "\n".join(rows)

# Display the concise summary string
summary_text

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                         trace_name                 timestamp       value
4683    IG01      trace.from_root.duration_mean 2021-03-09 16:55:00+08:00    0.820375
4684    IG01       trace.from_root.duration_p95 2021-03-09 16:55:00+08:00    1.367250
4685    IG01          trace.from_root.row_count 2021-03-09 16:55:00+08:00   16.000000
5079    IG01           trace.self.duration_mean 2021-03-09 16:57:00+08:00    0.698341
5199    IG01           trace.self.duration_mean 2021-03-09 16:58:00+08:00    0.853676
...      ...                                ...                       ...         ...
3387    MG01  trace.from_Tomcat03.duration_mean 2021-03-09 16:48:00+08:00    0.630342
5220    MG01  trace.from_Tomcat03.duration_mean 2021-03-09 16:58:00+08:00    0.761768
1363    MG01   trace.from_Tomcat03.duration_p95 2021-03-09 16:37:00+08:00    2.721350
5221    MG01   trace.from_Tomcat03.duration_p95 2021-03-09 16:58:00+08:00    3.145500
5222    MG01      trace.from_Tomcat03.row_count 2021-03-09 16:58:00+08:00  436.000000

[50 rows x 4 columns],      cmdb_id  total_anomaly_rows_in_window earliest_anomaly_timestamp
0       MG01                            81  2021-03-09 16:32:00+08:00
1       MG02                            84  2021-03-09 16:30:00+08:00
2    Mysql01                             0                        NaT
3    Mysql02                             0                        NaT
4   Tomcat03                            33  2021-03-09 16:31:00+08:00
5   Tomcat04                            33  2021-03-09 16:31:00+08:00
6       IG01                            33  2021-03-09 16:31:00+08:00
7   dockerA1                            33  2021-03-09 16:30:00+08:00
8   dockerA2                            36  2021-03-09 16:32:00+08:00
9   dockerB1                            44  2021-03-09 16:30:00+08:00
10  dockerB2                            33  2021-03-09 16:30:00+08:00)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

# Reuse existing variables where possible: df_trace, global_p95_trace, tz
import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Ensure df_trace loaded and has dt column (timezone-aware UTC+8)
if 'df_trace' not in globals():
    df_trace = pd.read_csv('trace.csv')
df_trace['timestamp'] = df_trace['timestamp'].astype(int)
if 'dt' not in df_trace.columns:
    df_trace['dt'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Compute or reuse global P95 per (cmdb_id, trace_name) over the full series (do NOT restrict to window)
if 'global_p95_trace' not in globals():
    global_p95_trace = df_trace.groupby(['cmdb_id','trace_name'], as_index=False)['value'].quantile(0.95).rename(columns={'value':'global_p95'})

# Define window (UTC+8)
start = pd.Timestamp('2021-03-09 16:30:00', tz=tz)
end = pd.Timestamp('2021-03-09 17:00:00', tz=tz)

# Restrict to window and components MG01 and MG02
components = ['MG01','MG02']
df_win = df_trace[(df_trace['dt'] >= start) & (df_trace['dt'] <= end) & (df_trace['cmdb_id'].isin(components))].copy()

# Join global p95 (computed globally) into windowed data
df_win = df_win.merge(global_p95_trace, on=['cmdb_id','trace_name'], how='left')

# Identify anomaly rows where value >= global_p95 (only if global_p95 notna)
df_win['is_anomaly'] = False
mask_has_p95 = df_win['global_p95'].notna()
df_win.loc[mask_has_p95, 'is_anomaly'] = df_win.loc[mask_has_p95, 'value'] >= df_win.loc[mask_has_p95, 'global_p95']

# Keep only anomaly rows
anoms = df_win[df_win['is_anomaly']].copy()

# If no anomalies, prepare empty outputs
if anoms.empty:
    intervals_display = pd.DataFrame(columns=[
        'cmdb_id','trace_name','interval_start_timestamp','interval_end_timestamp',
        'length_in_minutes','max_value_in_interval','global_p95','severity_ratio'
    ])
    comp_summary = pd.DataFrame({
        'cmdb_id': components,
        'total_fault_intervals_detected': [0]*len(components),
        'longest_interval_length': [0]*len(components),
        'earliest_interval_start': [pd.NaT]*len(components)
    })
else:
    # For grouping consecutive-by-minute sequences per (cmdb_id, trace_name)
    anoms = anoms.sort_values(['cmdb_id','trace_name','dt']).reset_index(drop=True)

    # Compute minute-diff between consecutive anomaly rows within each (cmdb_id, trace_name)
    anoms['dt_diff_min'] = anoms.groupby(['cmdb_id','trace_name'])['dt'].diff().dt.total_seconds().div(60).fillna(0)

    # Start a new interval when dt_diff_min != 1
    anoms['new_interval'] = (anoms['dt_diff_min'] != 1).astype(int)
    # Cumulative sum of new_interval within each group gives interval id
    anoms['interval_id'] = anoms.groupby(['cmdb_id','trace_name'])['new_interval'].cumsum()

    # Build interval-level aggregation
    interval_agg = anoms.groupby(['cmdb_id','trace_name','interval_id']).agg(
        interval_start_timestamp = ('dt', 'min'),
        interval_end_timestamp = ('dt', 'max'),
        length_in_minutes = ('dt', 'count'),
        max_value_in_interval = ('value', 'max'),
        global_p95 = ('global_p95', 'first')
    ).reset_index(drop=False)

    # Compute severity_ratio, handle global_p95==0 -> inf, NaN remain NaN
    def sev_ratio(row):
        gp = row['global_p95']
        mv = row['max_value_in_interval']
        if pd.isna(gp):
            return np.nan
        if gp == 0:
            return np.inf
        return (mv - gp) / gp

    interval_agg['severity_ratio'] = interval_agg.apply(sev_ratio, axis=1)

    # Select and sort intervals: length desc then severity desc
    intervals_display = interval_agg[[
        'cmdb_id','trace_name','interval_start_timestamp','interval_end_timestamp',
        'length_in_minutes','max_value_in_interval','global_p95','severity_ratio'
    ]].sort_values(['length_in_minutes','severity_ratio'], ascending=[False, False]).head(20)

    # Per-component summary: total_fault_intervals_detected, longest_interval_length, earliest_interval_start
    comp_summary = interval_agg.groupby('cmdb_id').agg(
        total_fault_intervals_detected = ('length_in_minutes','count'),
        longest_interval_length = ('length_in_minutes','max'),
        earliest_interval_start = ('interval_start_timestamp','min')
    ).reset_index()

    # Ensure all requested components are present
    comp_summary = pd.DataFrame({'cmdb_id': components}).merge(comp_summary, on='cmdb_id', how='left')
    comp_summary['total_fault_intervals_detected'] = comp_summary['total_fault_intervals_detected'].fillna(0).astype(int)
    comp_summary['longest_interval_length'] = comp_summary['longest_interval_length'].fillna(0).astype(int)
    # earliest_interval_start may remain NaT for components with no intervals

# Final compact outputs (bounded)
intervals_display, comp_summary

```
Out[6]:
```
# Summarize the detected fault intervals for MG01 and MG02 as a plain-English string
# Reuse previous results: `intervals_display` and `comp_summary`
import pandas as pd

if 'intervals_display' not in globals() or 'comp_summary' not in globals():
    raise RuntimeError("Required variables not found in kernel state. Re-run the interval detection step first.")

top_intervals = intervals_display.head(5).copy()
comp = comp_summary.copy()

lines = []
lines.append("Fault-interval summary for MG01 and MG02 (window 2021-03-09 16:30→17:00 +08):")
lines.append("")
# Component-level summary
for _, r in comp.iterrows():
    start_ts = r['earliest_interval_start']
    start_str = start_ts.isoformat() if pd.notna(start_ts) else "none"
    lines.append(f"- {r['cmdb_id']}: detected {int(r['total_fault_intervals_detected'])} fault intervals; "
                 f"longest = {int(r['longest_interval_length'])} min; earliest start = {start_str}")
lines.append("")
lines.append("Key observations:")
lines.append("- All detected intervals in the returned set are very short bursts (length_in_minutes = 2).")
lines.append("- MG02 has slightly more intervals (77) than MG01 (63), with earliest anomalies at ~16:30–16:32.")
lines.append("- Fault intervals recur across many trace types (row_count and duration metrics), indicating brief spikes in inter-service activity/latency.")
lines.append("")
lines.append("Top example intervals (up to 5):")
for _, r in top_intervals.iterrows():
    lines.append(
        f"  - {r['cmdb_id']} | {r['trace_name']} | {r['interval_start_timestamp']} → {r['interval_end_timestamp']} "
        f"| length={int(r['length_in_minutes'])} min | max={r['max_value_in_interval']} | global_p95={r['global_p95']} "
        f"| severity_ratio={r['severity_ratio']:.3f}"
    )
lines.append("")
lines.append("Recommended next steps:")
lines.append("- Prioritize investigation of MG01/MG02 around the earliest timestamps (check network, load, and logs).")
lines.append("- Because intervals are short and pervasive across many trace names, correlate these bursts with network spikes and docker/app metrics to determine cause (transient queuing vs sustained issue).")

summary_text = "\n".join(lines)
summary_text

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id                         trace_name  interval_start_timestamp    interval_end_timestamp  length_in_minutes  max_value_in_interval  global_p95  severity_ratio
21     MG01      trace.from_dockerA1.row_count 2021-03-09 16:58:00+08:00 2021-03-09 16:59:00+08:00                  2             163.000000  110.300000        0.477788
12     MG01  trace.from_Tomcat04.duration_mean 2021-03-09 16:57:00+08:00 2021-03-09 16:58:00+08:00                  2               0.727111    0.583042        0.247099
31     MG01      trace.from_dockerB1.row_count 2021-03-09 16:57:00+08:00 2021-03-09 16:58:00+08:00                  2             527.000000  430.600000        0.223874
90     MG02      trace.from_dockerA1.row_count 2021-03-09 16:46:00+08:00 2021-03-09 16:47:00+08:00                  2              87.000000   72.400000        0.201657
87     MG02  trace.from_dockerA1.duration_mean 2021-03-09 16:40:00+08:00 2021-03-09 16:41:00+08:00                  2               0.158429    0.132038        0.199877
52     MG01        trace.to_dockerA2.row_count 2021-03-09 16:56:00+08:00 2021-03-09 16:57:00+08:00                  2             437.000000  365.150000        0.196768
3      MG01  trace.from_Tomcat02.duration_mean 2021-03-09 16:57:00+08:00 2021-03-09 16:58:00+08:00                  2               0.662769    0.557373        0.189095
32     MG01  trace.from_dockerB2.duration_mean 2021-03-09 16:34:00+08:00 2021-03-09 16:35:00+08:00                  2               0.117664    0.099025        0.188227
1      MG01   trace.from_Tomcat01.duration_p95 2021-03-09 16:57:00+08:00 2021-03-09 16:58:00+08:00                  2               3.162000    2.690395        0.175292
0      MG01  trace.from_Tomcat01.duration_mean 2021-03-09 16:57:00+08:00 2021-03-09 16:58:00+08:00                  2               0.699709    0.612071        0.143183
39     MG01           trace.self.duration_mean 2021-03-09 16:57:00+08:00 2021-03-09 16:58:00+08:00                  2               0.440949    0.393161        0.121548
35     MG01      trace.from_dockerB2.row_count 2021-03-09 16:57:00+08:00 2021-03-09 16:58:00+08:00                  2             483.000000  433.450000        0.114315
26     MG01      trace.from_dockerA2.row_count 2021-03-09 16:57:00+08:00 2021-03-09 16:58:00+08:00                  2             125.000000  112.450000        0.111605
133    MG02        trace.to_dockerB1.row_count 2021-03-09 16:48:00+08:00 2021-03-09 16:49:00+08:00                  2             271.000000  245.000000        0.106122
57     MG01        trace.to_dockerB1.row_count 2021-03-09 16:58:00+08:00 2021-03-09 16:59:00+08:00                  2             449.000000  406.250000        0.105231
11     MG01      trace.from_Tomcat03.row_count 2021-03-09 16:58:00+08:00 2021-03-09 16:59:00+08:00                  2             471.000000  427.450000        0.101883
6      MG01      trace.from_Tomcat02.row_count 2021-03-09 16:57:00+08:00 2021-03-09 16:58:00+08:00                  2             489.000000  446.400000        0.095430
62     MG01        trace.to_dockerB2.row_count 2021-03-09 16:58:00+08:00 2021-03-09 16:59:00+08:00                  2             438.000000  402.350000        0.088604
127    MG02        trace.to_dockerA2.row_count 2021-03-09 16:46:00+08:00 2021-03-09 16:47:00+08:00                  2             227.000000  208.700000        0.087686
80     MG02      trace.from_Tomcat03.row_count 2021-03-09 16:48:00+08:00 2021-03-09 16:49:00+08:00                  2             278.000000  256.300000        0.084666,   cmdb_id  total_fault_intervals_detected  longest_interval_length   earliest_interval_start
0    MG01                              63                        2 2021-03-09 16:32:00+08:00
1    MG02                              77                        2 2021-03-09 16:30:00+08:00)```
```

In [ ]:
# In[7]:

# Reuse existing kernel state where possible to be efficient
import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Ensure df_metric is loaded
if 'df_metric' not in globals():
    df_metric = pd.read_csv('metric.csv')

# Ensure timestamp is integer and a timezone-aware dt column exists (UTC+8)
df_metric['timestamp'] = df_metric['timestamp'].astype(int)
if 'dt' not in df_metric.columns:
    df_metric['dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Compute or reuse global P95 per (cmdb_id, kpi_name) over the full series (must be global, not windowed)
if 'global_p95' not in globals():
    global_p95 = df_metric.groupby(['cmdb_id', 'kpi_name'])['value'].quantile(0.95).rename('global_p95').reset_index()

# Define window (UTC+8)
start = pd.Timestamp('2021-03-09 16:30:00', tz=tz)
end = pd.Timestamp('2021-03-09 17:00:00', tz=tz)

# Restrict to window and to components MG01 and MG02
components = ['MG01','MG02']
df_win = df_metric[(df_metric['dt'] >= start) & (df_metric['dt'] <= end) & (df_metric['cmdb_id'].isin(components))].copy()

# Join global P95 (computed globally) into the windowed dataframe
df_win = df_win.merge(global_p95, on=['cmdb_id','kpi_name'], how='left')

# Identify anomaly rows where value >= global_p95 (only when global_p95 is present)
mask_has_p95 = df_win['global_p95'].notna()
df_win['is_anomaly'] = False
df_win.loc[mask_has_p95, 'is_anomaly'] = df_win.loc[mask_has_p95, 'value'] >= df_win.loc[mask_has_p95, 'global_p95']

# Aggregate per (cmdb_id, kpi_name) within the window
agg = df_win.groupby(['cmdb_id', 'kpi_name']).agg(
    total_points_in_window = ('value', 'count'),
    anomaly_point_count = ('is_anomaly', 'sum'),
    max_value_in_window = ('value', 'max')
).reset_index()

# Earliest anomaly timestamp per group (NaT if none)
earliest_anom = df_win[df_win['is_anomaly']].groupby(['cmdb_id','kpi_name'])['dt'].min().rename('earliest_anomaly_timestamp').reset_index()

# Combine with global_p95 and earliest anomaly
summary = agg.merge(global_p95, on=['cmdb_id','kpi_name'], how='left').merge(earliest_anom, on=['cmdb_id','kpi_name'], how='left')

# Keep only groups with at least one anomaly (value >= global_p95) as the user requested identification of such rows
summary = summary[summary['anomaly_point_count'] > 0].copy()

# Compute severity_ratio, handling global_p95 == 0
def compute_severity_vec(gp, mv):
    # vectorized-safe function
    out = np.full(len(gp), np.nan, dtype=float)
    mask_notna = ~pd.isna(gp)
    mask_zero = mask_notna & (gp == 0)
    mask_pos = mask_notna & (gp != 0)
    out[mask_zero] = np.inf
    out[mask_pos] = (mv[mask_pos] - gp[mask_pos]) / gp[mask_pos]
    return out

summary['severity_ratio'] = compute_severity_vec(summary['global_p95'].values, summary['max_value_in_window'].values)

# Select and order columns as requested and sort by severity_ratio desc then anomaly_point_count desc
summary_display = summary[[
    'cmdb_id', 'kpi_name', 'global_p95', 'total_points_in_window',
    'anomaly_point_count', 'earliest_anomaly_timestamp', 'max_value_in_window', 'severity_ratio'
]].sort_values(['severity_ratio', 'anomaly_point_count'], ascending=[False, False]).head(20)

# Display the compact summary table (max 20 rows)
summary_display

```
Out[7]:
```
Summary of results (MG01 & MG02, window 2021-03-09 16:30 → 17:00 +08):

What I did
- Computed global P95 per (cmdb_id, kpi_name) using the full metric series (not windowed).
- Filtered metric rows to the incident window and kept only MG01 and MG02.
- Flagged points in the window with value >= that global P95 and aggregated per (cmdb_id, kpi_name).

Key findings
- Many of the top flagged KPIs have global_p95 == 0, which makes severity_ratio infinite. Those KPIs are mostly network/error counters and local-disk metrics.
- Examples from the top results:
  - MG02 / OSLinux-..._NETWORK_TCP_CLOSE_WAIT: global_p95=0.0, total points=21, anomaly_point_count=21, earliest anomaly 2021-03-09 16:30, max=2 → severity_ratio = inf
  - Multiple MG01 network/disk/process KPIs have global_p95=0 and anomaly_point_count == total_points_in_window (10–12 points each), earliest anomalies around 16:30–16:33, severity_ratio = inf.
- In many groups every point in the window was flagged (anomaly_point_count == total_points_in_window). This commonly happens when global_p95 is zero (baseline zeros) and current values are >= 0.

Interpretation
- A global P95 of 0 means the metric is usually zero or near-zero historically. Using ratio vs P95 for such metrics produces infinite/meaningless severity scores.
- Some spikes are real (e.g., MG02 TCP CLOSE_WAIT with max=2) but many flagged groups are merely non-zero occurrences vs a zero baseline rather than very large absolute deviations.

Recommended next steps
- Inspect raw metric values for the highlighted KPIs (confirm whether values are significant in absolute terms).
- Check logs/traces on MG01 and MG02 around the earliest anomaly timestamps (≈16:30–16:33).
- For KPIs with global_p95 == 0, use absolute thresholds or different anomaly logic (not ratio to P95) to avoid misleading infinite severity.
- Prioritize investigation of MG02 network TCP state and MG01 disk/network counters first, then correlate with application traces and system logs.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  global_p95  total_points_in_window  anomaly_point_count earliest_anomaly_timestamp  max_value_in_window  severity_ratio
162    MG02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT         0.0                      21                   21  2021-03-09 16:30:00+08:00                  2.0             inf
78     MG01       OSLinux-OSLinux_NETWORK_ens160_NETOutErrPrcc         0.0                      12                   12  2021-03-09 16:30:00+08:00                  0.0             inf
75     MG01         OSLinux-OSLinux_NETWORK_ens160_NETInErrPrc         0.0                      11                   11  2021-03-09 16:30:00+08:00                  0.0             inf
151    MG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps         0.0                      11                   11  2021-03-09 16:30:00+08:00                  0.0             inf
152    MG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead         0.0                      11                   11  2021-03-09 16:30:00+08:00                  0.0             inf
154    MG02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps         0.0                      11                   11  2021-03-09 16:32:00+08:00                  0.0             inf
170    MG02       OSLinux-OSLinux_NETWORK_ens160_NETOutErrPrcc         0.0                      11                   11  2021-03-09 16:32:00+08:00                  0.0             inf
175    MG02      OSLinux-OSLinux_PROCESS_PROCESS_PROCPPMemPerc         0.0                      11                   11  2021-03-09 16:30:00+08:00                  0.0             inf
12     MG01                    OSLinux-CPU_CPU-2_SingleCpuidle         0.0                      10                   10  2021-03-09 16:31:00+08:00                  0.0             inf
41     MG01  OSLinux-OSLinux_FILESYSTEM_-tomcat_FSAvailable...         0.0                      10                   10  2021-03-09 16:33:00+08:00                  0.0             inf
50     MG01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps         0.0                      10                   10  2021-03-09 16:31:00+08:00                  0.0             inf
51     MG01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead         0.0                      10                   10  2021-03-09 16:31:00+08:00                  0.0             inf
53     MG01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps         0.0                      10                   10  2021-03-09 16:33:00+08:00                  0.0             inf
59     MG01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps         0.0                      10                   10  2021-03-09 16:31:00+08:00                  0.0             inf
60     MG01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead         0.0                      10                   10  2021-03-09 16:31:00+08:00                  0.0             inf
62     MG01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps         0.0                      10                   10  2021-03-09 16:33:00+08:00                  0.0             inf
74     MG01            OSLinux-OSLinux_NETWORK_ens160_NETInErr         0.0                      10                   10  2021-03-09 16:33:00+08:00                  0.0             inf
77     MG01           OSLinux-OSLinux_NETWORK_ens160_NETOutErr         0.0                      10                   10  2021-03-09 16:33:00+08:00                  0.0             inf
83     MG01      OSLinux-OSLinux_PROCESS_PROCESS_PROCPPMemPerc         0.0                      10                   10  2021-03-09 16:31:00+08:00                  0.0             inf
86     MG01    OSLinux-OSLinux_SWAP_SWAP_SWPTotSwapUsedPercent         0.0                      10                   10  2021-03-09 16:31:00+08:00                  0.0             inf```
```

In [ ]:
# In[8]:

# Reuse existing kernel variables where possible
import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Ensure df_metric is loaded
if 'df_metric' not in globals():
    df_metric = pd.read_csv('metric.csv')

# Ensure timestamp is integer and timezone-aware dt column exists (UTC+8)
df_metric['timestamp'] = df_metric['timestamp'].astype(int)
if 'dt' not in df_metric.columns:
    df_metric['dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Compute global P95 per (cmdb_id, kpi_name) over the full series if not already present
if 'global_p95' not in globals():
    global_p95 = df_metric.groupby(['cmdb_id', 'kpi_name'], as_index=False)['value'].quantile(0.95).rename(columns={'value':'global_p95'})

# Define window and components
start = pd.Timestamp('2021-03-09 16:30:00', tz=tz)
end = pd.Timestamp('2021-03-09 17:00:00', tz=tz)
components = ['MG01','MG02']

# Filter metric rows to window and components
df_win = df_metric[(df_metric['dt'] >= start) & (df_metric['dt'] <= end) & (df_metric['cmdb_id'].isin(components))].copy()

# Join global_p95 (computed globally) into the windowed dataframe
df_win = df_win.merge(global_p95, on=['cmdb_id','kpi_name'], how='left')

# Keep only KPIs whose global_p95 > 0
df_win_pos = df_win[df_win['global_p95'] > 0].copy()

# Mark anomaly points where value >= global_p95
df_win_pos['is_anomaly'] = df_win_pos['value'] >= df_win_pos['global_p95']

# Extract only anomaly rows and detect consecutive-by-minute sequences per (cmdb_id,kpi_name)
anoms = df_win_pos[df_win_pos['is_anomaly']].copy()
anoms = anoms.sort_values(['cmdb_id','kpi_name','dt']).reset_index(drop=True)

if anoms.empty:
    # Empty outputs when no anomalies
    intervals_display = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','interval_start_timestamp','interval_end_timestamp',
        'length_in_minutes','max_value_in_interval','global_p95','severity_ratio'
    ])
    comp_summary = pd.DataFrame({
        'cmdb_id': components,
        'total_fault_intervals_detected': [0]*len(components),
        'longest_interval_length': [0]*len(components),
        'earliest_interval_start': [pd.NaT]*len(components)
    })
else:
    # Compute minute differences within each (cmdb_id,kpi_name)
    anoms['dt_diff_min'] = anoms.groupby(['cmdb_id','kpi_name'])['dt'].diff().dt.total_seconds().div(60)
    # Start new interval when difference is not exactly 1 minute (or NaN for first row)
    anoms['new_interval'] = (anoms['dt_diff_min'] != 1).astype(int)
    # Cumulative sum within each series to assign interval ids
    anoms['interval_id'] = anoms.groupby(['cmdb_id','kpi_name'])['new_interval'].cumsum()

    # Aggregate intervals
    interval_agg = anoms.groupby(['cmdb_id','kpi_name','interval_id'], as_index=False).agg(
        interval_start_timestamp = ('dt', 'min'),
        interval_end_timestamp = ('dt', 'max'),
        length_in_minutes = ('dt', 'count'),
        max_value_in_interval = ('value', 'max'),
        global_p95 = ('global_p95', 'first')
    )

    # Compute severity_ratio (global_p95 > 0 here)
    interval_agg['severity_ratio'] = (interval_agg['max_value_in_interval'] - interval_agg['global_p95']) / interval_agg['global_p95']

    # Prepare interval display sorted by length desc then severity desc, limit 20
    intervals_display = interval_agg[[
        'cmdb_id','kpi_name','interval_start_timestamp','interval_end_timestamp',
        'length_in_minutes','max_value_in_interval','global_p95','severity_ratio'
    ]].sort_values(['length_in_minutes','severity_ratio'], ascending=[False, False]).head(20)

    # Per-component summary
    comp_summary = interval_agg.groupby('cmdb_id').agg(
        total_fault_intervals_detected = ('length_in_minutes','count'),
        longest_interval_length = ('length_in_minutes','max'),
        earliest_interval_start = ('interval_start_timestamp','min')
    ).reset_index()

    # Ensure all requested components are present
    comp_summary = pd.DataFrame({'cmdb_id': components}).merge(comp_summary, on='cmdb_id', how='left')
    comp_summary['total_fault_intervals_detected'] = comp_summary['total_fault_intervals_detected'].fillna(0).astype(int)
    comp_summary['longest_interval_length'] = comp_summary['longest_interval_length'].fillna(0).astype(int)
    # earliest_interval_start may remain NaT for components with no intervals

# Final compact outputs (intervals_display limited to top 20, and comp_summary one row per component)
intervals_display, comp_summary

```
Out[8]:
```
# Summarize the detected metric fault intervals for MG01 and MG02 in plain English
import pandas as pd

# Use existing variables from kernel: intervals_display, comp_summary
if 'intervals_display' not in globals() or 'comp_summary' not in globals():
    raise RuntimeError("Required results not found in kernel state. Re-run the interval detection step first.")

intervals = intervals_display.copy()
comp = comp_summary.copy()

lines = []
lines.append("Metric fault-interval summary for MG01 and MG02 (window 2021-03-09 16:30→17:00 +08):")
lines.append("")
# Component-level summary
for _, r in comp.iterrows():
    start = r['earliest_interval_start']
    start_str = start.isoformat() if pd.notna(start) else "none"
    lines.append(f"- {r['cmdb_id']}: detected {int(r['total_fault_intervals_detected'])} intervals; longest = {int(r['longest_interval_length'])} min; earliest start = {start_str}")
lines.append("")
# Observations
lines.append("Key observations:")
lines.append("- Both MG01 and MG02 show a very large number of short fault intervals (MG01: 393, MG02: 385).")
lines.append("- The longest detected intervals span the entire window (31 minutes) for JVM memory max / NoHeapMemory metrics on both hosts, but severity_ratio is 0 for those rows (max == global_p95).")
lines.append("- Most other intervals are short (2–4 minutes) with small positive severity ratios (e.g., JVM HeapMemoryUsed on MG01 shows ~6.9% above P95 for a 2-minute interval).")
lines.append("")
# Top example intervals
lines.append("Top example intervals (up to 5):")
for _, r in intervals.head(5).iterrows():
    start = r['interval_start_timestamp'].isoformat()
    end = r['interval_end_timestamp'].isoformat()
    lines.append(f"  - {r['cmdb_id']} | {r['kpi_name']} | {start} → {end} | length={int(r['length_in_minutes'])} min | max={r['max_value_in_interval']} | global_p95={r['global_p95']} | severity_ratio={r['severity_ratio']:.6g}")
lines.append("")
# Recommendations
lines.append("Interpretation and next steps:")
lines.append("- Many intervals reflect metrics sitting at or near their historical P95 (severity_ratio ≈ 0) or frequent short spikes; this may indicate baseline high values rather than one-off extreme spikes.")
lines.append("- Prioritize metrics with meaningful positive severity_ratio (e.g., HeapMemoryUsed on MG01) and investigate corresponding logs/traces around their earliest interval timestamps.")
lines.append("- For metrics with global_p95 == max (severity 0) or many repeated short intervals, consider examining baseline behavior and adjusting anomaly rules (absolute thresholds or different percentiles).")

summary_text = "\n".join(lines)
summary_text

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id                                           kpi_name  interval_start_timestamp    interval_end_timestamp  length_in_minutes  max_value_in_interval    global_p95  severity_ratio
0      MG01           JVM-Memory_7779_JVM_Memory_HeapMemoryMax 2021-03-09 16:30:00+08:00 2021-03-09 17:00:00+08:00                 31           2.040136e+09  2.040136e+09        0.000000
4      MG01        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed 2021-03-09 16:30:00+08:00 2021-03-09 17:00:00+08:00                 31           5.069049e+08  5.069049e+08        0.000000
393    MG02           JVM-Memory_7779_JVM_Memory_HeapMemoryMax 2021-03-09 16:30:00+08:00 2021-03-09 17:00:00+08:00                 31           2.040136e+09  2.040136e+09        0.000000
397    MG02        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed 2021-03-09 16:30:00+08:00 2021-03-09 17:00:00+08:00                 31           9.471166e+08  9.471166e+08        0.000000
720    MG02       OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc 2021-03-09 16:31:00+08:00 2021-03-09 16:34:00+08:00                  4           9.800000e+01  9.800000e+01        0.000000
317    MG01             OSLinux-OSLinux_MEMORY_MEMORY_CacheMem 2021-03-09 16:31:00+08:00 2021-03-09 16:33:00+08:00                  3           3.987000e+03  3.987000e+03        0.000000
328    MG01       OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc 2021-03-09 16:31:00+08:00 2021-03-09 16:33:00+08:00                  3           9.700000e+01  9.700000e+01        0.000000
336    MG01       OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc 2021-03-09 16:56:00+08:00 2021-03-09 16:58:00+08:00                  3           9.700000e+01  9.700000e+01        0.000000
348    MG01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies 2021-03-09 16:31:00+08:00 2021-03-09 16:33:00+08:00                  3           2.000000e+00  2.000000e+00        0.000000
358    MG01          OSLinux-OSLinux_PROCESS_PROCESS_PROCPPMem 2021-03-09 16:31:00+08:00 2021-03-09 16:33:00+08:00                  3           5.800000e+01  5.800000e+01        0.000000
379    MG01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount 2021-03-09 16:31:00+08:00 2021-03-09 16:33:00+08:00                  3           3.000000e+00  3.000000e+00        0.000000
3      MG01          JVM-Memory_7779_JVM_Memory_HeapMemoryUsed 2021-03-09 16:37:00+08:00 2021-03-09 16:38:00+08:00                  2           1.010654e+09  9.451582e+08        0.069296
394    MG02         JVM-Memory_7779_JVM_Memory_HeapMemoryUsage 2021-03-09 16:53:00+08:00 2021-03-09 16:54:00+08:00                  2           5.955940e+01  5.765100e+01        0.033103
265    MG01  OSLinux-OSLinux_FILE_-tmp-zabbix_agentd.log_Fi... 2021-03-09 16:53:00+08:00 2021-03-09 16:54:00+08:00                  2           2.269000e-01  2.269000e-01        0.000000
318    MG01             OSLinux-OSLinux_MEMORY_MEMORY_CacheMem 2021-03-09 16:35:00+08:00 2021-03-09 16:36:00+08:00                  2           3.987000e+03  3.987000e+03        0.000000
319    MG01             OSLinux-OSLinux_MEMORY_MEMORY_CacheMem 2021-03-09 16:38:00+08:00 2021-03-09 16:39:00+08:00                  2           3.987000e+03  3.987000e+03        0.000000
320    MG01             OSLinux-OSLinux_MEMORY_MEMORY_CacheMem 2021-03-09 16:41:00+08:00 2021-03-09 16:42:00+08:00                  2           3.987000e+03  3.987000e+03        0.000000
321    MG01             OSLinux-OSLinux_MEMORY_MEMORY_CacheMem 2021-03-09 16:44:00+08:00 2021-03-09 16:45:00+08:00                  2           3.987000e+03  3.987000e+03        0.000000
322    MG01             OSLinux-OSLinux_MEMORY_MEMORY_CacheMem 2021-03-09 16:47:00+08:00 2021-03-09 16:48:00+08:00                  2           3.987000e+03  3.987000e+03        0.000000
323    MG01             OSLinux-OSLinux_MEMORY_MEMORY_CacheMem 2021-03-09 16:50:00+08:00 2021-03-09 16:51:00+08:00                  2           3.987000e+03  3.987000e+03        0.000000,   cmdb_id  total_fault_intervals_detected  longest_interval_length   earliest_interval_start
0    MG01                             393                       31 2021-03-09 16:30:00+08:00
1    MG02                             385                       31 2021-03-09 16:30:00+08:00)```
```

In [ ]:
# In[9]:

# Reuse existing kernel state where possible
import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Ensure df_trace is loaded and has timezone-aware dt column (UTC+8)
if 'df_trace' not in globals():
    df_trace = pd.read_csv('trace.csv')
df_trace['timestamp'] = df_trace['timestamp'].astype(int)
if 'dt' not in df_trace.columns:
    df_trace['dt'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Compute global P95 per (cmdb_id, trace_name) over the full series (do NOT filter by window)
if 'global_p95_trace' not in globals():
    global_p95_trace = df_trace.groupby(['cmdb_id', 'trace_name'], as_index=False)['value'].quantile(0.95).rename(columns={'value':'global_p95'})

# 2) Filter trace rows to the incident window and restrict to Tomcat components and trace_name patterns
start = pd.Timestamp('2021-03-09 16:30:00', tz=tz)
end = pd.Timestamp('2021-03-09 17:00:00', tz=tz)
tomcats = ['Tomcat01','Tomcat02','Tomcat03','Tomcat04']

mask_window = (df_trace['dt'] >= start) & (df_trace['dt'] <= end)
mask_comp = df_trace['cmdb_id'].isin(tomcats)
mask_name = df_trace['trace_name'].str.contains(r'to_Tomcat|from_Tomcat|trace.self', regex=True)

df_tomcat_win = df_trace[mask_window & mask_comp & mask_name].copy()

# 3) Join global_p95 (computed globally) into the windowed dataframe
df_tomcat_win = df_tomcat_win.merge(global_p95_trace, on=['cmdb_id','trace_name'], how='left')

# 4) Identify anomaly points where value >= global_p95 (only when global_p95 not null)
df_tomcat_win['is_anomaly'] = False
mask_has_p95 = df_tomcat_win['global_p95'].notna()
df_tomcat_win.loc[mask_has_p95, 'is_anomaly'] = df_tomcat_win.loc[mask_has_p95, 'value'] >= df_tomcat_win.loc[mask_has_p95, 'global_p95']

# 5) Aggregate per (cmdb_id, trace_name)
agg = df_tomcat_win.groupby(['cmdb_id','trace_name']).agg(
    total_points_in_window = ('value','count'),
    anomaly_point_count = ('is_anomaly','sum'),
    max_value_in_window = ('value','max')
).reset_index()

# earliest anomaly timestamp per group
earliest = df_tomcat_win[df_tomcat_win['is_anomaly']].groupby(['cmdb_id','trace_name'])['dt'].min().rename('earliest_anomaly_timestamp').reset_index()

# combine with global_p95 and earliest anomaly
summary = agg.merge(global_p95_trace, on=['cmdb_id','trace_name'], how='left').merge(earliest, on=['cmdb_id','trace_name'], how='left')

# 6) compute severity_ratio (inf if global_p95 == 0)
def compute_sev(gp, mv):
    if pd.isna(gp):
        return np.nan
    if gp == 0:
        return np.inf
    return (mv - gp) / gp

summary['severity_ratio'] = summary.apply(lambda r: compute_sev(r['global_p95'], r['max_value_in_window']), axis=1)

# 7) Keep only groups that have at least one anomaly point (anomaly_point_count > 0)
summary = summary[summary['anomaly_point_count'] > 0].copy()

# 8) Select requested columns and sort; limit to top 20 rows
summary_display = summary[[
    'cmdb_id','trace_name','global_p95','total_points_in_window',
    'anomaly_point_count','earliest_anomaly_timestamp','max_value_in_window','severity_ratio'
]].sort_values(['severity_ratio','anomaly_point_count'], ascending=[False, False]).head(20)

# Return compact result
summary_display

```
Out[9]:
```
# Summarize the trace analysis results in plain English using existing `summary_display`
import pandas as pd

if 'summary_display' not in globals():
    raise RuntimeError("summary_display not found in kernel state. Re-run the Tomcat trace aggregation step first.")

df = summary_display.copy()

lines = []
lines.append("Tomcat trace anomaly summary (window 2021-03-09 16:30→17:00 +08):")
lines.append(f"- Returned rows: {len(df)} (filtered to Tomcat components and trace.self / to_Tomcat / from_Tomcat patterns)")
lines.append("- All flagged groups are Tomcat traces (trace.self and duration/row_count variants).")
lines.append("")
# Top by severity_ratio
top = df.sort_values(['severity_ratio','anomaly_point_count'], ascending=[False, False]).head(3)
lines.append("Top relative deviations:")
for _, r in top.iterrows():
    lines.append(f"  - {r['cmdb_id']} | {r['trace_name']} | global_p95={r['global_p95']} | max={r['max_value_in_window']} | severity_ratio={r['severity_ratio']:.3f} | anomalies={int(r['anomaly_point_count'])} | earliest={r['earliest_anomaly_timestamp']}")
lines.append("")
lines.append("Key observations:")
lines.append("- The largest relative increase is Tomcat03 trace.self.duration_mean (~25% above its global P95).")
lines.append("- All entries have very small counts of anomaly points (mostly 2 points in the window), indicating brief spikes rather than sustained degradation.")
lines.append("- Other notable items: small positive increases in trace.self.duration_p95 and trace.self.row_count across Tomcat nodes (few-percent changes).")
lines.append("")
lines.append("Recommendation:")
lines.append("- Inspect traces/logs around the reported earliest timestamps (e.g., 16:48 and 16:57) for Tomcat03/Tomcat02/Tomcat04/Tomcat01.")
lines.append("- Correlate with host metrics (CPU, network, GC) to determine whether these brief latency/row_count spikes impacted user requests.")
lines.append("- Because anomalies are short and small in relative terms (except Tomcat03 duration_mean), prioritize Tomcat03 for initial investigation.")

summary_text = "\n".join(lines)
summary_text

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                trace_name   global_p95  total_points_in_window  anomaly_point_count earliest_anomaly_timestamp  max_value_in_window  severity_ratio
6   Tomcat03  trace.self.duration_mean     0.103620                      31                    2  2021-03-09 16:48:00+08:00             0.129291        0.247748
3   Tomcat02  trace.self.duration_mean     0.099951                      30                    2  2021-03-09 16:57:00+08:00             0.121482        0.215412
9   Tomcat04  trace.self.duration_mean     0.107164                      30                    2  2021-03-09 16:48:00+08:00             0.126858        0.183774
0   Tomcat01  trace.self.duration_mean     0.110878                      30                    2  2021-03-09 16:57:00+08:00             0.121428        0.095148
11  Tomcat04      trace.self.row_count  3160.850000                      30                    2  2021-03-09 16:34:00+08:00          3315.000000        0.048769
2   Tomcat01      trace.self.row_count  3248.300000                      30                    2  2021-03-09 16:39:00+08:00          3367.000000        0.036542
8   Tomcat03      trace.self.row_count  3318.000000                      31                    2  2021-03-09 16:48:00+08:00          3401.000000        0.025015
4   Tomcat02   trace.self.duration_p95     1.006550                      30                    2  2021-03-09 16:57:00+08:00             1.030000        0.023297
1   Tomcat01   trace.self.duration_p95     1.016357                      30                    2  2021-03-09 16:57:00+08:00             1.036050        0.019376
10  Tomcat04   trace.self.duration_p95     1.012150                      30                    2  2021-03-09 16:48:00+08:00             1.031000        0.018624
7   Tomcat03   trace.self.duration_p95     1.023300                      31                    2  2021-03-09 16:50:00+08:00             1.035600        0.012020
5   Tomcat02      trace.self.row_count  3315.750000                      30                    2  2021-03-09 16:34:00+08:00          3336.000000        0.006107```
```

In [ ]:
# In[10]:

# Reuse existing environment variables where possible
import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load df_log if not present
if 'df_log' not in globals():
    df_log = pd.read_csv('log.csv')

# Ensure timestamp is integer and create timezone-aware datetime column (UTC+8)
df_log['timestamp'] = df_log['timestamp'].astype(int)
if 'dt' not in df_log.columns:
    df_log['dt'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Compute global P95 per (cmdb_id, log_name) using the full series (global thresholds)
global_p95_log = df_log.groupby(['cmdb_id', 'log_name'], as_index=False)['value'].quantile(0.95).rename(columns={'value':'global_p95'})

# 2) Filter log rows to the incident window and the requested components
start = pd.Timestamp('2021-03-09 16:30:00', tz=tz)
end = pd.Timestamp('2021-03-09 17:00:00', tz=tz)
components = ['MG01','MG02','Tomcat01','Tomcat02','Tomcat03','Tomcat04','Mysql01','Mysql02','IG01',
              'dockerA1','dockerA2','dockerB1','dockerB2']
df_win_log = df_log[(df_log['dt'] >= start) & (df_log['dt'] <= end) & (df_log['cmdb_id'].isin(components))].copy()

# 3) Join global P95 into the windowed logs
df_win_log = df_win_log.merge(global_p95_log, on=['cmdb_id','log_name'], how='left')

# 4) Determine anomaly rows:
#    condition A: value >= global_p95 (only when global_p95 is present)
#    condition B: log_name contains 'error' (case-insensitive) and value > 0
mask_has_p95 = df_win_log['global_p95'].notna()
cond_a = mask_has_p95 & (df_win_log['value'] >= df_win_log['global_p95'])
cond_b = df_win_log['log_name'].str.contains('error', case=False, na=False) & (df_win_log['value'] > 0)
df_win_log['is_anomaly'] = cond_a | cond_b

# 5) Aggregate per (cmdb_id, log_name)
agg_log = df_win_log.groupby(['cmdb_id','log_name']).agg(
    global_p95 = ('global_p95','first'),
    total_points_in_window = ('value','count'),
    anomaly_point_count = ('is_anomaly','sum'),
    max_value_in_window = ('value','max')
).reset_index()

# 6) Earliest anomaly timestamp per group (if any)
earliest_anom_log = df_win_log[df_win_log['is_anomaly']].groupby(['cmdb_id','log_name'])['dt'].min().rename('earliest_anomaly_timestamp').reset_index()

# 7) Combine earliest timestamps
agg_log = agg_log.merge(earliest_anom_log, on=['cmdb_id','log_name'], how='left')

# 8) Compute severity_ratio = (max - global_p95)/global_p95 ; handle global_p95 == 0 -> inf ; NaN stays NaN
def compute_severity(gp, mv):
    if pd.isna(gp):
        return np.nan
    if gp == 0:
        return np.inf
    return (mv - gp) / gp

agg_log['severity_ratio'] = agg_log.apply(lambda r: compute_severity(r['global_p95'], r['max_value_in_window']), axis=1)

# 9) Keep only groups that have at least one anomaly (anomaly_point_count > 0)
agg_log_flagged = agg_log[agg_log['anomaly_point_count'] > 0].copy()

# 10) Sort by severity_ratio desc then anomaly_point_count desc, limit to top 20 rows for display
detail_display = agg_log_flagged[[
    'cmdb_id','log_name','global_p95','total_points_in_window',
    'anomaly_point_count','earliest_anomaly_timestamp','max_value_in_window','severity_ratio'
]].sort_values(['severity_ratio','anomaly_point_count'], ascending=[False, False]).head(20)

# 11) Per-component summary: count of distinct log_name groups flagged and earliest anomaly timestamp per component
comp_summary = agg_log_flagged.groupby('cmdb_id').agg(
    total_anomaly_log_groups = ('log_name','nunique'),
    earliest_log_anomaly_timestamp = ('earliest_anomaly_timestamp','min')
).reset_index()

# Ensure all requested components are present in the summary (fill zeros/NaT where missing)
comp_summary_all = pd.DataFrame({'cmdb_id': components}).merge(comp_summary, on='cmdb_id', how='left')
comp_summary_all['total_anomaly_log_groups'] = comp_summary_all['total_anomaly_log_groups'].fillna(0).astype(int)
# earliest timestamp may remain NaT for components with no anomalies

# Final compact outputs: detail_display (<=20 rows) and comp_summary_all (one row per requested component)
detail_display, comp_summary_all

```
Out[10]:
```
# Summarize the log analysis results (plain English) using existing variables
summary_lines = []
summary_lines.append("Log anomaly summary (global P95 computed on full series; window 2021-03-09 16:30→17:00 +08):")
summary_lines.append("")
summary_lines.append("What was done:")
summary_lines.append("- Computed global P95 per (cmdb_id, log_name) using the full log series.")
summary_lines.append("- Filtered logs to the incident window and to the requested components.")
summary_lines.append("- Flagged anomaly rows where value >= global_p95 OR (log_name contains 'error' and value > 0).")
summary_lines.append("")
summary_lines.append("Key findings (top flagged log groups):")
summary_lines.append("- Several Tomcat nodes have log.error_count flagged across the entire window (Tomcat01/02/03/04).")
summary_lines.append("  Note: these are flagged because value >= global_p95 and global_p95 == 0 (current values are 0, so 0 >= 0).")
summary_lines.append("- Real, non-zero deviations worth inspecting:")
summary_lines.append("  - Tomcat02 log.gc_pause_max: global_p95=0.09, max=0.14, severity_ratio≈0.56 (5 anomaly points).")
summary_lines.append("  - Tomcat01 log.gc_pause_max: global_p95=0.09, max=0.10, severity_ratio≈0.11 (6 anomaly points).")
summary_lines.append("  - Mysql01 log.row_count: global_p95=13.25, max=16, severity_ratio≈0.21 (2 anomaly points).")
summary_lines.append("- Small percent increases in log.row_count and log.gc_pause_max appear across Tomcat nodes (mostly 2–6 anomaly points each).")
summary_lines.append("")
summary_lines.append("Component-level summary:")
# Use comp_summary_all if available to list counts and earliest timestamps concisely
if 'comp_summary_all' in globals():
    for _, r in comp_summary_all.iterrows():
        ts = r['earliest_log_anomaly_timestamp']
        ts_str = ts.isoformat() if pd.notna(ts) else "none"
        summary_lines.append(f"- {r['cmdb_id']}: total_anomaly_log_groups={int(r['total_anomaly_log_groups'])}, earliest_anomaly={ts_str}")
else:
    summary_lines.append("- Component summary not available in kernel state.")

summary_lines.append("")
summary_lines.append("Interpretation and recommendations:")
summary_lines.append("- Many groups are flagged solely because global_p95 == 0 (baseline zeros). That makes severity_ratio infinite and is misleading.")
summary_lines.append("- For log.error_count and other zero-baseline logs, prefer absolute thresholds or require value > 0 for error semantics, not ratio to P95.")
summary_lines.append("- Prioritize investigating Tomcat GC pauses (Tomcat02 and Tomcat01) and the Mysql01 row_count increases around the reported earliest timestamps (~16:30–16:36).")
summary_lines.append("- Correlate these log anomalies with traces and host metrics (CPU, memory, disk, network) to determine impact and root cause.")

summary_text = "\n".join(summary_lines)
summary_text

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id          log_name  global_p95  total_points_in_window  anomaly_point_count earliest_anomaly_timestamp  max_value_in_window  severity_ratio
2   Tomcat01   log.error_count      0.0000                      31                   31  2021-03-09 16:30:00+08:00                 0.00             inf
8   Tomcat03   log.error_count      0.0000                      31                   31  2021-03-09 16:30:00+08:00                 0.00             inf
11  Tomcat04   log.error_count      0.0000                      31                   31  2021-03-09 16:30:00+08:00                 0.00             inf
5   Tomcat02   log.error_count      0.0000                      30                   30  2021-03-09 16:30:00+08:00                 0.00             inf
0    Mysql01   log.error_count      0.0000                      26                   26  2021-03-09 16:30:00+08:00                 0.00             inf
6   Tomcat02  log.gc_pause_max      0.0900                      30                    5  2021-03-09 16:33:00+08:00                 0.14        0.555556
1    Mysql01     log.row_count     13.2500                      26                    2  2021-03-09 16:56:00+08:00                16.00        0.207547
3   Tomcat01  log.gc_pause_max      0.0900                      22                    6  2021-03-09 16:30:00+08:00                 0.10        0.111111
12  Tomcat04  log.gc_pause_max      0.0995                      22                    2  2021-03-09 16:49:00+08:00                 0.11        0.105528
9   Tomcat03  log.gc_pause_max      0.1095                      22                    2  2021-03-09 16:45:00+08:00                 0.12        0.095890
4   Tomcat01     log.row_count    475.0000                      31                    2  2021-03-09 16:36:00+08:00               508.00        0.069474
13  Tomcat04     log.row_count    463.5000                      31                    2  2021-03-09 16:36:00+08:00               493.00        0.063646
10  Tomcat03     log.row_count    504.5000                      31                    2  2021-03-09 16:48:00+08:00               522.00        0.034688
7   Tomcat02     log.row_count      2.0000                      30                    7  2021-03-09 16:33:00+08:00                 2.00        0.000000,      cmdb_id  total_anomaly_log_groups earliest_log_anomaly_timestamp
0       MG01                         0                            NaT
1       MG02                         0                            NaT
2   Tomcat01                         3      2021-03-09 16:30:00+08:00
3   Tomcat02                         3      2021-03-09 16:30:00+08:00
4   Tomcat03                         3      2021-03-09 16:30:00+08:00
5   Tomcat04                         3      2021-03-09 16:30:00+08:00
6    Mysql01                         2      2021-03-09 16:30:00+08:00
7    Mysql02                         0                            NaT
8       IG01                         0                            NaT
9   dockerA1                         0                            NaT
10  dockerA2                         0                            NaT
11  dockerB1                         0                            NaT
12  dockerB2                         0                            NaT)```
```

In [ ]:
# In[11]:

# Reuse existing dataframes and variables where possible; compute missing globals as needed
import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load CSVs if not already in kernel
if 'df_metric' not in globals():
    df_metric = pd.read_csv('metric.csv')
if 'df_trace' not in globals():
    df_trace = pd.read_csv('trace.csv')
if 'df_log' not in globals():
    df_log = pd.read_csv('log.csv')

# Ensure timestamp ints and timezone-aware dt columns (UTC+8)
for df in (df_metric, df_trace, df_log):
    df['timestamp'] = df['timestamp'].astype(int)
    if 'dt' not in df.columns:
        df['dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Components of interest
components = ['MG01','MG02','Tomcat01','Tomcat02','Tomcat03','Tomcat04',
              'Mysql01','Mysql02','IG01','dockerA1','dockerA2','dockerB1','dockerB2']

# 1) Compute global P95 thresholds over full series (do NOT filter by window)
if 'global_p95' not in globals():
    global_p95 = df_metric.groupby(['cmdb_id','kpi_name'], as_index=False)['value'].quantile(0.95).rename(columns={'value':'global_p95'})
if 'global_p95_trace' not in globals():
    global_p95_trace = df_trace.groupby(['cmdb_id','trace_name'], as_index=False)['value'].quantile(0.95).rename(columns={'value':'global_p95'})
if 'global_p95_log' not in globals():
    global_p95_log = df_log.groupby(['cmdb_id','log_name'], as_index=False)['value'].quantile(0.95).rename(columns={'value':'global_p95'})

# 2) Filter rows to incident window and restrict to components
start = pd.Timestamp('2021-03-09 16:30:00', tz=tz)
end = pd.Timestamp('2021-03-09 17:00:00', tz=tz)

m_win = df_metric[(df_metric['dt'] >= start) & (df_metric['dt'] <= end) & (df_metric['cmdb_id'].isin(components))].copy()
t_win = df_trace[(df_trace['dt'] >= start) & (df_trace['dt'] <= end) & (df_trace['cmdb_id'].isin(components))].copy()
l_win = df_log[(df_log['dt'] >= start) & (df_log['dt'] <= end) & (df_log['cmdb_id'].isin(components))].copy()

# 3) Join thresholds and detect anomalies per row

# Metrics: join global_p95 (by cmdb_id,kpi_name); anomaly if global_p95>0 AND value >= global_p95
m_win = m_win.merge(global_p95, on=['cmdb_id','kpi_name'], how='left')
m_win['is_metric_anom'] = (m_win['global_p95'].notna()) & (m_win['global_p95'] > 0) & (m_win['value'] >= m_win['global_p95'])

# Traces: join global_p95_trace; anomaly if global_p95 exists and value >= global_p95
t_win = t_win.merge(global_p95_trace, on=['cmdb_id','trace_name'], how='left')
t_win['is_trace_anom'] = (t_win['global_p95'].notna()) & (t_win['value'] >= t_win['global_p95'])

# Logs: join global_p95_log; anomaly if (global_p95 exists and value >= global_p95) OR (log_name contains 'error' AND value > 0)
l_win = l_win.merge(global_p95_log, on=['cmdb_id','log_name'], how='left')
cond_a = (l_win['global_p95'].notna()) & (l_win['value'] >= l_win['global_p95'])
cond_b = l_win['log_name'].str.contains('error', case=False, na=False) & (l_win['value'] > 0)
l_win['is_log_anom'] = cond_a | cond_b

# 4) Per-component aggregates: earliest timestamp and counts
# Metrics
m_agg = m_win[m_win['is_metric_anom']].groupby('cmdb_id').agg(
    earliest_metric_anomaly = ('dt','min'),
    metric_anomaly_count = ('is_metric_anom','sum')
).reset_index()

# Traces
t_agg = t_win[t_win['is_trace_anom']].groupby('cmdb_id').agg(
    earliest_trace_anomaly = ('dt','min'),
    trace_anomaly_count = ('is_trace_anom','sum')
).reset_index()

# Logs
l_agg = l_win[l_win['is_log_anom']].groupby('cmdb_id').agg(
    earliest_log_anomaly = ('dt','min'),
    log_anomaly_count = ('is_log_anom','sum')
).reset_index()

# 5) Combine into one compact summary table (one row per requested component)
summary_df = pd.DataFrame({'cmdb_id': components})
summary_df = summary_df.merge(m_agg, on='cmdb_id', how='left') \
                       .merge(t_agg, on='cmdb_id', how='left') \
                       .merge(l_agg, on='cmdb_id', how='left')

# Fill counts with 0 where missing; keep earliest timestamps as NaT if none
summary_df['metric_anomaly_count'] = summary_df['metric_anomaly_count'].fillna(0).astype(int)
summary_df['trace_anomaly_count'] = summary_df['trace_anomaly_count'].fillna(0).astype(int)
summary_df['log_anomaly_count'] = summary_df['log_anomaly_count'].fillna(0).astype(int)

# Keep column order as requested
summary_df = summary_df[[
    'cmdb_id',
    'earliest_metric_anomaly','metric_anomaly_count',
    'earliest_trace_anomaly','trace_anomaly_count',
    'earliest_log_anomaly','log_anomaly_count'
]]

# Final compact output (one row per specified component)
summary_df

```
Out[11]:
```
# Summarize the combined anomaly overview for the requested components as a plain-English string
# Uses previously computed `summary_df` from kernel state
if 'summary_df' not in globals():
    raise RuntimeError("summary_df not found. Re-run the aggregation step first.")

rows = []
rows.append("Combined anomaly summary for window 2021-03-09 16:30→17:00 +08 (one line per component):")
for _, r in summary_df.iterrows():
    cm = r['cmdb_id']
    m_ts = r['earliest_metric_anomaly']
    t_ts = r['earliest_trace_anomaly']
    l_ts = r['earliest_log_anomaly']
    m_count = int(r['metric_anomaly_count'])
    t_count = int(r['trace_anomaly_count'])
    l_count = int(r['log_anomaly_count'])
    m_ts_s = m_ts.isoformat() if pd.notna(m_ts) else "none"
    t_ts_s = t_ts.isoformat() if pd.notna(t_ts) else "none"
    l_ts_s = l_ts.isoformat() if pd.notna(l_ts) else "none"
    rows.append(f"- {cm}: metric earliest={m_ts_s} (count={m_count}); trace earliest={t_ts_s} (count={t_count}); log earliest={l_ts_s} (count={l_count})")

rows.append("")
rows.append("High-level observations:")
rows.append("- MG01 and MG02: many metric anomalies (513 and 484 rows) and many trace anomalies (81 and 84), but no log anomalies flagged in this window. Investigate MG hosts' metrics and traces first.")
rows.append("- Tomcat01–04: each has metric anomalies (~478–511) and trace anomalies (~33–34); additionally each Tomcat shows log anomalies (30–42 flagged log groups) starting ~16:30. Correlate Tomcat logs/traces with host metrics.")
rows.append("- Mysql01/Mysql02: very high metric anomaly counts (811/808). Mysql01 also shows log anomalies (28 groups) at ~16:30; Mysql02 has no log anomalies. Check MySQL host metrics and the logged events on Mysql01.")
rows.append("- IG01: metric and trace anomalies present (metric earliest 16:30, trace earliest 16:31) but no log anomalies flagged.")
rows.append("- Docker nodes: dockerA1/A2/dockerB1/dockerB2 show notable trace anomalies (33–44 rows) but few metric anomalies; dockerA2 has no metric anomalies in the window. Focus on inter-service traces for docker nodes.")
rows.append("")
rows.append("Recommended next steps:")
rows.append("- Prioritize MG01/MG02 and MySQL hosts for metric+trace investigation (high anomaly counts).")
rows.append("- For Tomcats, focus on correlating log.error and GC pause spikes with the trace/metric anomalies around ~16:30–16:36.")
rows.append("- For docker nodes and IG01, inspect traces and network/CPU metrics to determine if those trace spikes correspond to transient network/latency events.")
rows.append("- Note: many metric/log groups use global P95==0; treat zero-baseline metrics/logs differently (use absolute thresholds) to avoid misleading infinite severity scores.")

summary_text = "\n".join(rows)
summary_text

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id   earliest_metric_anomaly  metric_anomaly_count    earliest_trace_anomaly  trace_anomaly_count      earliest_log_anomaly  log_anomaly_count
0       MG01 2021-03-09 16:30:00+08:00                   513 2021-03-09 16:32:00+08:00                   81                       NaT                  0
1       MG02 2021-03-09 16:30:00+08:00                   484 2021-03-09 16:30:00+08:00                   84                       NaT                  0
2   Tomcat01 2021-03-09 16:30:00+08:00                   478 2021-03-09 16:32:00+08:00                   34 2021-03-09 16:30:00+08:00                 39
3   Tomcat02 2021-03-09 16:30:00+08:00                   502 2021-03-09 16:31:00+08:00                   33 2021-03-09 16:30:00+08:00                 42
4   Tomcat03 2021-03-09 16:30:00+08:00                   511 2021-03-09 16:31:00+08:00                   33 2021-03-09 16:30:00+08:00                 35
5   Tomcat04 2021-03-09 16:30:00+08:00                   484 2021-03-09 16:31:00+08:00                   33 2021-03-09 16:30:00+08:00                 35
6    Mysql01 2021-03-09 16:30:00+08:00                   811                       NaT                    0 2021-03-09 16:30:00+08:00                 28
7    Mysql02 2021-03-09 16:30:00+08:00                   808                       NaT                    0                       NaT                  0
8       IG01 2021-03-09 16:30:00+08:00                   442 2021-03-09 16:31:00+08:00                   33                       NaT                  0
9   dockerA1 2021-03-09 16:48:00+08:00                     4 2021-03-09 16:30:00+08:00                   33                       NaT                  0
10  dockerA2                       NaT                     0 2021-03-09 16:32:00+08:00                   36                       NaT                  0
11  dockerB1 2021-03-09 16:33:00+08:00                     7 2021-03-09 16:30:00+08:00                   44                       NaT                  0
12  dockerB2 2021-03-09 16:40:00+08:00                     6 2021-03-09 16:30:00+08:00                   33                       NaT                  0```
```

In [ ]:
# In[12]:

# Reuse existing kernel state where possible and follow the rules
import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load dataframes if not present
if 'df_metric' not in globals():
    df_metric = pd.read_csv('metric.csv')
if 'df_trace' not in globals():
    df_trace = pd.read_csv('trace.csv')
if 'df_log' not in globals():
    df_log = pd.read_csv('log.csv')

# Ensure timestamp ints and timezone-aware datetime columns (UTC+8)
for df in (df_metric, df_trace, df_log):
    df['timestamp'] = df['timestamp'].astype(int)
    if 'dt' not in df.columns:
        df['dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Components and window
components = ['MG01','MG02','Tomcat01','Tomcat02','Tomcat03','Tomcat04',
              'Mysql01','Mysql02','IG01','dockerA1','dockerA2','dockerB1','dockerB2']
start = pd.Timestamp('2021-03-09 16:30:00', tz=tz)
end = pd.Timestamp('2021-03-09 17:00:00', tz=tz)

# 1) Compute global P95 per series (if not already computed in kernel)
if 'global_p95' not in globals():
    global_p95 = df_metric.groupby(['cmdb_id','kpi_name'], as_index=False)['value'].quantile(0.95).rename(columns={'value':'global_p95'})
if 'global_p95_trace' not in globals():
    global_p95_trace = df_trace.groupby(['cmdb_id','trace_name'], as_index=False)['value'].quantile(0.95).rename(columns={'value':'global_p95'})
if 'global_p95_log' not in globals():
    global_p95_log = df_log.groupby(['cmdb_id','log_name'], as_index=False)['value'].quantile(0.95).rename(columns={'value':'global_p95'})

# 2) Filter rows to the incident window and components
m_win = df_metric[(df_metric['dt'] >= start) & (df_metric['dt'] <= end) & (df_metric['cmdb_id'].isin(components))].copy()
t_win = df_trace[(df_trace['dt'] >= start) & (df_trace['dt'] <= end) & (df_trace['cmdb_id'].isin(components))].copy()
l_win = df_log[(df_log['dt'] >= start) & (df_log['dt'] <= end) & (df_log['cmdb_id'].isin(components))].copy()

# 3) Metrics: join global_p95, compute severity_ratio, keep rows where global_p95>0 and severity_ratio >= 0.5
m_win = m_win.merge(global_p95, on=['cmdb_id','kpi_name'], how='left')
m_win['severity_ratio'] = np.where(m_win['global_p95'] > 0, (m_win['value'] - m_win['global_p95']) / m_win['global_p95'], np.nan)
m_high = m_win[(m_win['global_p95'] > 0) & (m_win['severity_ratio'] >= 0.5)].copy()
m_high['signal_type'] = 'metric'
m_high['series_name'] = m_high['kpi_name']
m_high = m_high[['cmdb_id','signal_type','series_name','dt','value','global_p95','severity_ratio']].rename(columns={'dt':'timestamp'})

# 3) Traces: join global_p95_trace, compute severity_ratio, keep rows where global_p95>0 and severity_ratio >= 0.5
t_win = t_win.merge(global_p95_trace, on=['cmdb_id','trace_name'], how='left')
t_win['severity_ratio'] = np.where(t_win['global_p95'] > 0, (t_win['value'] - t_win['global_p95']) / t_win['global_p95'], np.nan)
t_high = t_win[(t_win['global_p95'] > 0) & (t_win['severity_ratio'] >= 0.5)].copy()
t_high['signal_type'] = 'trace'
t_high['series_name'] = t_high['trace_name']
t_high = t_high[['cmdb_id','signal_type','series_name','dt','value','global_p95','severity_ratio']].rename(columns={'dt':'timestamp'})

# 3) Logs: join global_p95_log, compute severity_ratio where possible.
l_win = l_win.merge(global_p95_log, on=['cmdb_id','log_name'], how='left')
# compute severity ratio for rows with global_p95>0
l_win['severity_ratio'] = np.where(l_win['global_p95'] > 0, (l_win['value'] - l_win['global_p95']) / l_win['global_p95'], np.nan)
# condition A: high relative to P95 (value > gp and severity >=0.5)
cond_a = (l_win['global_p95'] > 0) & (l_win['value'] > l_win['global_p95']) & (l_win['severity_ratio'] >= 0.5)
# condition B: error logs with value > 0
cond_b = l_win['log_name'].str.contains('error', case=False, na=False) & (l_win['value'] > 0)
l_high = l_win[cond_a | cond_b].copy()
l_high['signal_type'] = 'log'
l_high['series_name'] = l_high['log_name']
# For error rows where global_p95 is 0 or NaN, set severity_ratio to inf if gp==0 else keep computed
l_high['severity_ratio'] = l_high.apply(
    lambda r: (np.inf if (pd.isna(r['global_p95']) or r['global_p95'] == 0) and ('error' in str(r['series_name']).lower()) else r['severity_ratio']),
    axis=1
)
l_high = l_high[['cmdb_id','signal_type','series_name','dt','value','global_p95','severity_ratio']].rename(columns={'dt':'timestamp'})

# 4) Combine high-severity rows across types
combined = pd.concat([m_high, t_high, l_high], ignore_index=True, sort=False)
# Normalize timestamp column name and ensure timezone-aware dtype
combined['timestamp'] = pd.to_datetime(combined['timestamp']).dt.tz_convert(tz)

# B) Detailed high-severity anomaly list (max 20 rows), sort by severity_ratio desc then timestamp asc
detail_sorted = combined.sort_values(['severity_ratio','timestamp'], ascending=[False, True])
detail_display = detail_sorted[['cmdb_id','signal_type','series_name','timestamp','value','global_p95','severity_ratio']].head(20)

# A) Per-component earliest-high-severity-anomaly table (one row per component)
# For each component, pick the earliest timestamp among combined; if tie pick highest severity
if not combined.empty:
    combined_sorted_for_earliest = combined.sort_values(['cmdb_id','timestamp','severity_ratio'], ascending=[True, True, False])
    earliest_per_comp = combined_sorted_for_earliest.groupby('cmdb_id', as_index=False).first()
    # Keep only required columns and ensure all components present
    earliest_per_comp = earliest_per_comp[['cmdb_id','signal_type','series_name','timestamp','value','global_p95','severity_ratio']]
else:
    earliest_per_comp = pd.DataFrame(columns=['cmdb_id','signal_type','series_name','timestamp','value','global_p95','severity_ratio'])

# Ensure one row per requested component (fill missing with nulls)
per_component = pd.DataFrame({'cmdb_id': components}).merge(earliest_per_comp, on='cmdb_id', how='left')

# Rename columns to match requested names for A
per_component = per_component.rename(columns={
    'signal_type':'signal_type',
    'series_name':'series_name',
    'timestamp':'earliest_anomaly_timestamp',
    'value':'value_at_earliest'
})[[
    'cmdb_id','signal_type','series_name','earliest_anomaly_timestamp','value_at_earliest','global_p95','severity_ratio'
]]

# Keep outputs compact and bounded
per_component, detail_display

```
Out[12]:
```
Summary of high-severity anomalies (thresholds = global P95 computed on the full series; window = 2021-03-09 16:30 → 17:00 +08)

A) Per-component earliest high-severity event (one line highlights)
- MG01 — metric: OSLinux-CPU_CPU_CPUWio — earliest 2021-03-09 16:30 — value 3.4582, global_p95 1.38348, severity_ratio ≈ 1.50 (150% above P95).
- MG02 — metric: OSLinux-CPU_CPU_CPUWio — earliest 2021-03-09 16:30 — value 0.9490, global_p95 0.50292, severity_ratio ≈ 0.89.
- Tomcat01 — metric: local-disk DSKTps — earliest 2021-03-09 16:33 — value 1.0, global_p95 0.5, severity_ratio = 1.0.
- Tomcat02 — metric: local-disk DSKPer... — earliest 2021-03-09 16:38 — value 0.2, global_p95 0.1, severity_ratio = 1.0.
- Tomcat03 — metric: local-disk DSKBps — earliest 2021-03-09 16:30 — value 5.0, global_p95 3.0, severity_ratio ≈ 0.67.
- Tomcat04 — metric: NETWORK_TCP-CLOSE-WAIT — earliest 2021-03-09 16:31 — value 1.0, global_p95 0.05, severity_ratio = 19.0 (very large relative jump).
- Mysql01 — metric: Mysql-MySQL_3306_Com Insert — earliest 2021-03-09 16:30 — value 0.2667, global_p95 0.013335, severity_ratio = 19.0.
- Mysql02 — metric: Mysql-MySQL_3306_SlowQueries — earliest 2021-03-09 16:30 — value 17.4754, global_p95 6.732135, severity_ratio ≈ 1.60.
- IG01 — metric: local-disk DSKTps — earliest 2021-03-09 16:40 — value 1.0, global_p95 0.5, severity_ratio = 1.0.
- dockerA2 — trace: trace.to_MG02.duration_mean — earliest 2021-03-09 16:55 — value 0.22125, global_p95 0.132121, severity_ratio ≈ 0.675.
- dockerA1, dockerB1, dockerB2 — no qualifying high-severity (nulls shown).

Note: the table above shows each component’s earliest detected high-severity signal_type (metric/trace/log), the series name, earliest timestamp, the observed value, global P95 and severity_ratio.

B) Detailed high-severity anomaly list (top examples from the returned set)
- Mysql01 | metric | Mysql-MySQL_3306_Com Insert | 2021-03-09 16:30 | value 0.2667 | gp95 0.013335 | severity 19.0
- Tomcat04 | metric | NETWORK_TCP-CLOSE-WAIT | 2021-03-09 16:31 | value 1.0 | gp95 0.05 | severity 19.0
- Mysql02 | metric | CurrentSQLMaxRunningTime / LongestTrxActiveTime | 2021-03-09 16:43 | value 1.0 | gp95 0.05 | severity 19.0
- MG02 | metric | NETWORK_TCP-FIN-WAIT | 2021-03-09 16:56 | value 10.0 | gp95 0.5 | severity 19.0
- Tomcat03 | metric | NETWORK_TCP-CLOSE-WAIT | 2021-03-09 17:00 | value 1.0 | gp95 0.05 | severity 19.0
- MG01 | metric | NETWORK_TCP-CLOSE-WAIT | 2021-03-09 16:52 | value 1.0 | gp95 0.10 | severity 9.0
- Several high-severity events are local-disk IO spikes and TCP state counters; others are MySQL-specific spikes (disk read/IO and MySQL metrics).
- Also included: trace high-severity rows such as dockerA2 → MG02 duration (severity ≈ 0.675), and disk throughput spikes on Mysql/MG/IG hosts (severity ratios 2.3x, 3x, etc).

Important interpretation notes
- Many very large severity_ratio values (e.g., 19.0) result from extremely small global_p95 baselines (global_p95 ≪ 1). These indicate large relative change but may reflect small absolute counts. Treat such ratios carefully—absolute impact matters.
- The high-severity filter used: severity_ratio ≥ 0.5 (≥50% above global P95). For logs we additionally included any error log with value > 0.
- Some components (dockerA1, dockerB1, dockerB2) had no rows meeting the high-severity criteria.

Recommended next steps
1. Investigate the items with the largest severity_ratio and earliest timestamps first (Mysql01, Tomcat04, MG02, etc.), but confirm absolute significance (e.g., 1 → 0.05 is a big ratio but may be low absolute).
2. Correlate these high-severity metric/trace spikes with logs, GC pauses, and host resource metrics (CPU, disk I/O, network) around the reported earliest timestamps.
3. For series with near-zero baseline P95, consider using absolute thresholds or minimum-value guards to avoid over-prioritizing tiny-count spikes.

If you want, I can:
- show the full per-component table A exactly as returned,
- or expand the detailed list B to display up to 50 rows for deeper inspection.

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id signal_type                                        series_name earliest_anomaly_timestamp  value_at_earliest  global_p95  severity_ratio
0       MG01      metric                             OSLinux-CPU_CPU_CPUWio  2021-03-09 16:30:00+08:00            3.45820    1.383480        1.499639
1       MG02      metric                             OSLinux-CPU_CPU_CPUWio  2021-03-09 16:30:00+08:00            0.94900    0.502920        0.886980
2   Tomcat01      metric     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps  2021-03-09 16:33:00+08:00            1.00000    0.500000        1.000000
3   Tomcat02      metric  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...  2021-03-09 16:38:00+08:00            0.20000    0.100000        1.000000
4   Tomcat03      metric     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps  2021-03-09 16:30:00+08:00            5.00000    3.000000        0.666667
5   Tomcat04      metric     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT  2021-03-09 16:31:00+08:00            1.00000    0.050000       19.000000
6    Mysql01      metric                        Mysql-MySQL_3306_Com Insert  2021-03-09 16:30:00+08:00            0.26670    0.013335       19.000000
7    Mysql02      metric                       Mysql-MySQL_3306_SlowQueries  2021-03-09 16:30:00+08:00           17.47540    6.732135        1.595818
8       IG01      metric     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps  2021-03-09 16:40:00+08:00            1.00000    0.500000        1.000000
9   dockerA1         NaN                                                NaN                        NaT                NaN         NaN             NaN
10  dockerA2       trace                        trace.to_MG02.duration_mean  2021-03-09 16:55:00+08:00            0.22125    0.132121        0.674598
11  dockerB1         NaN                                                NaN                        NaT                NaN         NaN             NaN
12  dockerB2         NaN                                                NaN                        NaT                NaN         NaN             NaN,      cmdb_id signal_type                                        series_name                 timestamp       value  global_p95  severity_ratio
2    Mysql01      metric                        Mysql-MySQL_3306_Com Insert 2021-03-09 16:30:00+08:00    0.266700    0.013335       19.000000
10  Tomcat04      metric     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT 2021-03-09 16:31:00+08:00    1.000000    0.050000       19.000000
30   Mysql02      metric          Mysql-MySQL_3306_CurrentSQLMaxRunningTime 2021-03-09 16:43:00+08:00    1.000000    0.050000       19.000000
31   Mysql02      metric              Mysql-MySQL_3306_LongestTrxActiveTime 2021-03-09 16:43:00+08:00    1.000000    0.050000       19.000000
44      MG02      metric       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT 2021-03-09 16:56:00+08:00   10.000000    0.500000       19.000000
57  Tomcat03      metric     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT 2021-03-09 17:00:00+08:00    1.000000    0.050000       19.000000
43      MG01      metric     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT 2021-03-09 16:52:00+08:00    1.000000    0.100000        9.000000
33   Mysql02      metric     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps 2021-03-09 16:44:00+08:00    5.000000    1.250000        3.000000
29   Mysql01      metric  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer... 2021-03-09 16:43:00+08:00    0.800000    0.240000        2.333333
42      IG01      metric    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead 2021-03-09 16:50:00+08:00    6.400000    1.920000        2.333333
54   Mysql02      metric    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead 2021-03-09 17:00:00+08:00    1.733333    0.520000        2.333333
13   Mysql02      metric  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer... 2021-03-09 16:36:00+08:00    2.800000    0.840000        2.333333
41      IG01      metric    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps 2021-03-09 16:50:00+08:00    0.216667    0.065000        2.333333
53   Mysql02      metric    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps 2021-03-09 17:00:00+08:00    0.216667    0.065000        2.333333
17   Mysql01      metric    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead 2021-03-09 16:39:00+08:00   44.133300   13.706680        2.219839
40      IG01      metric  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer... 2021-03-09 16:50:00+08:00    1.900000    0.625000        2.040000
23   Mysql02      metric    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead 2021-03-09 16:40:00+08:00  380.000000  137.520000        1.763234
22   Mysql02      metric    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps 2021-03-09 16:40:00+08:00   10.683333    3.870000        1.760551
16   Mysql01      metric    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps 2021-03-09 16:39:00+08:00    0.816700    0.303320        1.692536
7    Mysql02      metric                       Mysql-MySQL_3306_SlowQueries 2021-03-09 16:30:00+08:00   17.475400    6.732135        1.595818)```
```